# 4단계 재개: 완료 경로를 건너뛰고 나머지만 실행

새 T4 또는 L4 GPU 세션에서 **코드 셀 1→5**를 실행합니다.
2번에 **repair_stage4_20261123_132010_265667.zip**을 업로드하세요.
기존 best.pt만 추가로 필요하며 학습·새 이미지 생성은 없습니다.

이 ZIP에는 첫 사례의 네 경로와 두 번째 사례의 일반 재확산이 완료돼 있습니다.
이 다섯 경로는 다시 계산하지 않습니다. 두 번째 사례의 미완료 보정부터 진행합니다.
중단된 경로의 중간 latent/controller가 저장되지 않았으므로 해당 경로는 1/15부터 재시작합니다.

각 경로 완료 시 전체 상태를 Drive에 자동 백업합니다. 4번 셀을 중지하면 중단 지점까지 백업하고,
5번에서 ZIP을 받을 수 있습니다. 연결 강제 종료 시에는 마지막 완료 경로의 Drive 백업을 사용하세요.
다음 재개에서도 이 노트북에 최신 checkpoint ZIP을 넣으면 됩니다.

옛 ZIP의 부분 사례에는 초기 latent hash가 없습니다. 과거 경로와 새 경로의 비트 단위 초기 상태
동일성을 소급해 증명할 수 없어 보고서에 legacy_initial_unrecorded=true로 명시합니다.
새로 계산하는 경로들은 같은 초기 latent를 공유하고 이를 저장해 이후 재개에서 재사용합니다.
GPU 종류나 패키지가 바뀐 이력도 부모 manifest와 versions.json에 남습니다.

In [ ]:
# 1. 설치·내장 코드 준비 (새 GPU 세션 권장)
import sys, subprocess, json, hashlib, gc, shutil, traceback
from pathlib import Path
from datetime import datetime
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.37.0', 'transformers==4.57.6', 'accelerate', 'safetensors',
    'timm', 'numpy', 'scipy', 'scikit-image', 'pillow', 'matplotlib'], check=True)
RUNTIME_HELPERS={'stroke_model/reference_model.py': '"""Reference-conditioned V2. Inputs: white ink on black, Bx1xHxW in [0,1].\n\nNo thresholding, CPU conversion or detached image features in the forward path.\nOld checkpoints are deliberately not compatible with this new architecture.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef block(cin, cout, stride=1):\n    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride=stride, padding=1),\n                         nn.GroupNorm(4, cout), nn.GELU())\n\n\ndef resize(x, size):\n    return F.interpolate(x, size=size, mode="bilinear", align_corners=False)\n\n\ndef warp(strokes, flow):\n    b, _, h, w = strokes.shape\n    ys = (torch.arange(h, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / h - 1\n    xs = (torch.arange(w, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / w - 1\n    y, x = torch.meshgrid(ys, xs, indexing="ij")\n    grid = torch.stack((x, y), -1)[None].expand(b, -1, -1, -1)\n    offset = torch.stack((flow[:, 0] * 2 / w, flow[:, 1] * 2 / h), -1)\n    return F.grid_sample(strokes, grid + offset, align_corners=False)\n\n\nclass ReferenceStrokeModel(nn.Module):\n    def __init__(self, image_size=128, width=64, pretrained=True,\n                 backbone=None, channels=None, channels_last=False, stroke_chunk_size=4):\n        super().__init__()\n        if width % 4:\n            raise ValueError("width must be divisible by 4")\n        mean, std = (.485, .456, .406), (.229, .224, .225)\n        if backbone is None:\n            import timm\n            backbone = timm.create_model(\n                "swin_tiny_patch4_window7_224", pretrained=pretrained,\n                features_only=True, out_indices=(0, 1, 2, 3),\n                in_chans=3, img_size=image_size, drop_path_rate=0.)\n            channels = backbone.feature_info.channels()\n            channels_last = True  # timm Swin features are NHWC.\n            mean = backbone.pretrained_cfg.get("mean", mean)\n            std = backbone.pretrained_cfg.get("std", std)\n        if not channels:\n            raise ValueError("Injected backbone requires its feature channels")\n        self.backbone = backbone\n        self.stroke_chunk_size = max(1, int(stroke_chunk_size))\n        self.channels_last = channels_last\n        self.backbone_frozen = True\n        self.backbone.requires_grad_(False)\n        self.backbone.eval()\n        self.register_buffer("rgb_mean", torch.tensor(mean).view(1, 3, 1, 1))\n        self.register_buffer("rgb_std", torch.tensor(std).view(1, 3, 1, 1))\n        self.lateral = nn.ModuleList(nn.Conv2d(c, width, 1) for c in channels)\n        self.smooth = nn.ModuleList(block(width, width) for _ in channels)\n        self.detail = block(1, width)  # Full-resolution branch for thin strokes.\n        self.fuse = block(width * 2, width)\n        self.flow_net = nn.Sequential(block(2, 16, 2), block(16, 32, 2),\n                                      nn.Conv2d(32, 2, 3, padding=1))\n        nn.init.zeros_(self.flow_net[-1].weight)\n        nn.init.zeros_(self.flow_net[-1].bias)\n        # Shared across stroke IDs; full glyph + selected stroke + xy context.\n        self.reference = nn.Sequential(block(4, 16, 2), block(16, width, 2))\n        self.head = nn.Sequential(block(width * 2 + 4, width),\n                                  nn.Conv2d(width, 1, 1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        if self.backbone_frozen:\n            self.backbone.eval()\n        return self\n\n    def forward(self, image, reference, strokes, active=None):\n        if image.ndim != 4 or image.shape[1] != 1 or reference.shape != image.shape:\n            raise ValueError("image/reference must have matching Bx1xHxW shapes")\n        b, _, h, w = image.shape\n        if strokes.ndim != 4 or strokes.shape[0] != b or strokes.shape[-2:] != (h, w):\n            raise ValueError("strokes must have shape BxKxHxW")\n        k = strokes.shape[1]\n        if active is None:\n            active = strokes.flatten(2).sum(-1) > 0  # Fixed reference metadata only.\n        if active.shape != (b, k) or active.dtype != torch.bool or not active.any(1).all():\n            raise ValueError("Each sample needs at least one active reference stroke")\n        rgb = (1 - image).expand(-1, 3, -1, -1)\n        feats = self.backbone((rgb - self.rgb_mean) / self.rgb_std)\n        if self.channels_last:\n            feats = [f.permute(0, 3, 1, 2) for f in feats]\n        pyramid = None\n        for i in reversed(range(len(feats))):\n            lateral = self.lateral[i](feats[i])\n            pyramid = lateral if pyramid is None else lateral + resize(pyramid, lateral.shape[-2:])\n            pyramid = self.smooth[i](pyramid)\n        features = self.fuse(torch.cat((resize(pyramid, (h, w)), self.detail(image)), 1))\n        flow = 12 * resize(self.flow_net(torch.cat((reference, image), 1)), (h, w)).tanh()\n        hints = warp(strokes, flow)\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=image.device, dtype=image.dtype),\n                                torch.linspace(-1, 1, w, device=image.device, dtype=image.dtype), indexing="ij")\n        xy = torch.stack((xx, yy))[None].expand(b, -1, -1, -1)\n        logits = self.score_strokes(features, reference, strokes, xy, flow, hints, image)\n        logits = logits.float().masked_fill(~active[:, :, None, None], -torch.inf)\n        probabilities = logits.softmax(1) * active[:, :, None, None]\n        probabilities = probabilities / probabilities.sum(1, keepdim=True)\n        return {"logits": logits, "probabilities": probabilities,\n                "masks": image * probabilities, "flow": flow, "hints": hints,\n                "active": active}\n\n    def score_strokes(self, features, reference, strokes, xy, flow, hints, image):\n        """Same head algebra, batching strokes and computing shared convolution once.\n\n        GroupNorm is per sample, so stroke batching does not mix stroke statistics.\n        Existing V2 state_dicts remain compatible; input gradients remain connected.\n        """\n        b, k, h, w = strokes.shape\n        width = features.shape[1]\n        conv = self.head[0][0]\n        shared_weight = torch.cat((conv.weight[:, :width], conv.weight[:, 2*width+1:]), 1)\n        shared = F.conv2d(torch.cat((features, image, xy), 1), shared_weight, conv.bias, padding=1)\n        outputs = []\n        for start in range(0, k, self.stroke_chunk_size):\n            n = min(self.stroke_chunk_size, k-start)\n            def repeat(t):\n                return t[:, None].expand(-1, n, -1, -1, -1).reshape(b*n, *t.shape[1:])\n            selected = strokes[:, start:start+n].reshape(b*n, 1, h, w)\n            r = self.reference(torch.cat((repeat(reference), selected, repeat(xy)), 1))\n            r = warp(resize(r, (h, w)), repeat(flow))\n            condition = torch.cat((r, hints[:, start:start+n].reshape(b*n, 1, h, w)), 1)\n            x = F.conv2d(condition, conv.weight[:, width:2*width+1], padding=1) + repeat(shared)\n            x = self.head[0][2](self.head[0][1](x))\n            outputs.append(self.head[1](x).reshape(b, n, h, w))\n        return torch.cat(outputs, 1)\n\n\ndef exclusive_masks(output, foreground):\n    """Export only: non-differentiable exact partition of the supplied foreground."""\n    expected = output[\'probabilities\'][:, :1].shape\n    if foreground.shape != expected:\n        raise ValueError(\'foreground must have shape Bx1xHxW\')\n    labels = output["probabilities"].argmax(1)\n    masks = F.one_hot(labels, output["probabilities"].shape[1]).permute(0, 3, 1, 2)\n    return masks.bool() & foreground.bool()\n\n\ndef stroke_moments(masks, eps=1e-6):\n    """Area fraction, centroid x/y and xx/xy/yy covariance in normalized coordinates.\n\n    Near-empty strokes have unreliable geometry: callers must mask/inspect them.\n    """\n    h, w = masks.shape[-2:]\n    y, x = torch.meshgrid(torch.linspace(-1, 1, h, device=masks.device, dtype=masks.dtype),\n                          torch.linspace(-1, 1, w, device=masks.device, dtype=masks.dtype), indexing="ij")\n    mass = masks.sum((-2, -1))\n    weights = masks / mass.clamp_min(eps)[..., None, None]\n    cx, cy = (weights * x).sum((-2, -1)), (weights * y).sum((-2, -1))\n    dx, dy = x - cx[..., None, None], y - cy[..., None, None]\n    return torch.stack((mass / (h * w), cx, cy,\n                        (weights * dx.square()).sum((-2, -1)),\n                        (weights * dx * dy).sum((-2, -1)),\n                        (weights * dy.square()).sum((-2, -1))), -1)\n\n\ndef partition_loss(output, labels):\n    """labels: BxHxW, 0..K-1 for owned pixels; -1 background/unknown.\n\n    Unknown crossing pixels are excluded, not assigned pseudo-certainty.\n    Soft Dice averages only annotated, nonempty target strokes.\n    """\n    logits = output["logits"]\n    valid = labels >= 0\n    if labels.shape != logits.shape[:1] + logits.shape[2:]:\n        raise ValueError("labels must have shape BxHxW")\n    if not valid.any() or (labels < -1).any() or (labels >= logits.shape[1]).any():\n        raise ValueError("Need annotated pixels with labels in -1..K-1")\n    target = F.one_hot(labels.clamp_min(0).long(), logits.shape[1]).permute(0, 3, 1, 2)\n    target = target * valid[:, None]\n    if (target.bool() & ~output["active"][:, :, None, None]).any():\n        raise ValueError("Target refers to an inactive reference stroke")\n    ce = F.cross_entropy(logits, labels.long(), ignore_index=-1)\n    pred = output["probabilities"] * valid[:, None]\n    mass = target.sum((-2, -1))\n    dice = (2 * (pred * target).sum((-2, -1)) + 1e-6) / (pred.sum((-2, -1)) + mass + 1e-6)\n    dice_loss = (1 - dice)[mass > 0].mean()\n    flow = output["flow"]\n    smooth = (flow[..., 1:, :] - flow[..., :-1, :]).square().mean()\n    smooth = smooth + (flow[..., :, 1:] - flow[..., :, :-1]).square().mean()\n    return {"loss": ce + dice_loss + .001 * smooth, "ce": ce,\n            "dice_loss": dice_loss, "flow_smoothness": smooth}\n', 'stroke_model/stroke_reference_fit.py': '"""Fit valid reference strokes to extracted stroke positions, not their defects.\n\nInference-only geometry: principal skeleton paths discard short side branches;\nstraight references remain straight, curves receive a regularized deformation.\nThe original reference centerline is deformed and re-rendered at the observed\nwidth. Output is a candidate for inspection, not a ground-truth reconstruction.\n"""\nfrom dataclasses import dataclass, asdict\nimport heapq\nimport numpy as np\nfrom scipy.ndimage import label, distance_transform_edt, map_coordinates, affine_transform,gaussian_filter1d\nfrom scipy.spatial import cKDTree\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass ReferenceFitConfig:\n    points: int=64\n    smoothness: float=12.\n    shape_prior: float=.15\n    max_displacement: float=16.\n    max_width: float=10.\n    min_width: float=1.\n    min_length_ratio: float=.65\n    max_length_ratio: float=1.6\n    junction_radius: float=2.5\n    junction_weight: float=8.\n    max_junction_shift: float=2.\n\n    def __post_init__(self):\n        if type(self.points) is not int or self.points<8:raise ValueError(\'At least eight path samples required\')\n        if any(not np.isfinite(v) or v<0 for v in asdict(self).values()):raise ValueError(\'Invalid fit configuration\')\n        if not self.min_width or self.max_width<self.min_width or not 0<self.min_length_ratio<self.max_length_ratio:\n            raise ValueError(\'Invalid width/length limits\')\n\n\ndef _unit(array):\n    array=np.asarray(array,np.float32)\n    if array.ndim!=2 or not np.isfinite(array).all() or array.min()<0 or array.max()>1:\n        raise ValueError(\'Expected a finite 2D ink mask in [0,1]\')\n    return array\n\n\ndef _component(mask,hint=None):\n    threshold=max(.1,min(.4,float(mask.max())*.4))\n    binary=mask>=threshold\n    labels,count=label(binary,np.ones((3,3)))\n    if not count:raise ValueError(\'Empty/low-confidence extracted stroke\')\n    distance=None\n    if hint is not None and hint.max()>.1:\n        distance=distance_transform_edt(hint<max(.1,float(hint.max())*.3))\n    scores=[]\n    for k in range(1,count+1):\n        region=labels==k;mass=float(mask[region].sum())\n        proximity=1. if distance is None else np.exp(-float(np.median(distance[region]))/20.)\n        scores.append(mass*proximity)\n    chosen=labels==(int(np.argmax(scores))+1)\n    return chosen,dict(components=count,selected_mass_fraction=float(mask[chosen].sum()/max(float(mask.sum()),1e-6)))\n\n\ndef _resample(path,n):\n    arc=np.r_[0,np.cumsum(np.linalg.norm(np.diff(path,axis=0),axis=1))]\n    if arc[-1]<1e-6:raise ValueError(\'Degenerate stroke path\')\n    keep=np.r_[True,np.diff(arc)>1e-8];arc=arc[keep];path=path[keep]\n    t=np.linspace(0,arc[-1],n)\n    return np.stack([np.interp(t,arc,path[:,i]) for i in range(2)],1)\n\n\ndef principal_path(binary,n=64):\n    """Longest geodesic path of the principal component; never reject spurs."""\n    skeleton=skeletonize(binary)\n    coordinates=np.argwhere(skeleton)\n    if len(coordinates)<2:raise ValueError(\'Insufficient centerline support\')\n    indices={tuple(yx):i for i,yx in enumerate(coordinates)}\n    graph=[[] for _ in coordinates]\n    for i,(y,x) in enumerate(coordinates):\n        for dy in (-1,0,1):\n            for dx in (-1,0,1):\n                if not (dx or dy):continue\n                j=indices.get((y+dy,x+dx))\n                if j is not None:graph[i].append((j,float(np.hypot(dx,dy))))\n    endpoints=[i for i,edges in enumerate(graph) if len(edges)==1]\n    def farthest(start):\n        distances=np.full(len(graph),np.inf);distances[start]=0.;parent=np.full(len(graph),-1,int)\n        queue=[(0.,start)]\n        while queue:\n            cost,i=heapq.heappop(queue)\n            if cost>distances[i]+1e-8:continue\n            for j,weight in graph[i]:\n                proposed=cost+weight\n                if proposed<distances[j]-1e-8:\n                    distances[j]=proposed;parent[j]=i;heapq.heappush(queue,(proposed,j))\n        candidates=endpoints if endpoints else list(range(len(graph)))\n        end=max(candidates,key=lambda j:distances[j] if np.isfinite(distances[j]) else -1)\n        return end,parent\n    first,_=farthest(endpoints[0] if endpoints else 0)\n    last,parent=farthest(first);order=[last]\n    while order[-1]!=first:\n        previous=parent[order[-1]]\n        if previous<0:raise ValueError(\'Disconnected centerline\')\n        order.append(int(previous))\n    path=coordinates[order,::-1].astype(float)\n    return _resample(path,n),dict(endpoints=len(endpoints),path_coverage=len(order)/len(coordinates),closed=len(endpoints)==0)\n\n\ndef _width(binary,path):\n    radius=distance_transform_edt(binary)\n    samples=map_coordinates(radius,[path[:,1],path[:,0]],order=1,mode=\'nearest\')\n    middle=samples[len(samples)//10:len(samples)-len(samples)//10]\n    return max(1.,float(np.median(2*middle-1)))\n\n\ndef render_reference(reference,original_path,target_path,width_scale):\n    """Render the deformed original centerline with the estimated stroke width.\n\n    Raster thin hints must not be stretched directly: their one-pixel staircase\n    and interpolation profile create serrated, gray thick strokes.\n    """\n    reference=_unit(reference);h,w=reference.shape\n    # Dense samples reduce nearest-segment seams while preserving reference caps.\n    # Keep normalized arc correspondence even when relative segment lengths change.\n    target=np.stack([np.interp(np.linspace(0,1,512),np.linspace(0,1,len(target_path)),target_path[:,i]) for i in range(2)],1)\n    yy,xx=np.mgrid[:h,:w];pixels=np.stack((xx.ravel(),yy.ravel()),1)\n    binary=reference>=max(.1,float(reference.max())*.3)\n    width=_width(binary,original_path)*width_scale\n    distance,_=cKDTree(target).query(pixels)\n    return np.clip(.5+(width/2-distance),0,1).reshape(h,w).astype(np.float32)\n\n\ndef _affine_fallback(reference,observed,config):\n    """Preserve a loop/branch prototype rather than cutting it into a path."""\n    def moments(mask):\n        y,x=np.nonzero(mask);xy=np.stack((x,y),1).astype(float)\n        center=xy.mean(0);extent=np.percentile(xy,95,axis=0)-np.percentile(xy,5,axis=0)\n        return center,np.maximum(extent,1)\n    source=reference>max(.1,float(reference.max())*.3)\n    rc,re=moments(source);tc,te=moments(observed)\n    shift=tc-rc;shift*=min(1.,config.max_displacement/max(np.linalg.norm(shift),1e-6))\n    scale=np.clip(te/re,.75,1.35)\n    # scipy uses y,x and maps output to input.\n    inverse=np.diag(1/scale[::-1]);offset=rc[::-1]-inverse@(rc+shift)[::-1]\n    result=affine_transform(reference,inverse,offset=offset,output_shape=reference.shape,order=1,mode=\'constant\')\n    return result.clip(0,1).astype(np.float32),dict(translation=shift.tolist(),scale=scale.tolist())\n\n\ndef fit_reference_stroke(reference,extracted,*,hint=None,config=None):\n    config=config or ReferenceFitConfig();reference=_unit(reference);extracted=_unit(extracted)\n    if reference.shape!=extracted.shape:raise ValueError(\'Reference/extraction shape mismatch\')\n    hint=reference if hint is None else _unit(hint)\n    if hint.shape!=reference.shape:raise ValueError(\'Hint shape mismatch\')\n    observed,component_info=_component(extracted,hint)\n    canonical,_=_component(reference)\n    original,path_info=principal_path(canonical,config.points)\n    metadata=dict(component_info,reference_path=path_info,warnings=[])\n    if path_info[\'closed\'] or path_info[\'path_coverage\']<.65 or (path_info[\'endpoints\']>2 and path_info[\'path_coverage\']<.9):\n        warped,transform=_affine_fallback(reference,observed,config)\n        metadata.update(method=\'affine_topology_preserving_fallback\',transform=transform)\n        metadata[\'warnings\'].append(\'Closed/branched prototype: affine fallback, no open-curve fit\')\n        return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n            reference_path=None,target_path=None,hint_path=None,width_scale=1.,metadata=metadata)\n    try:\n        hint_binary,_=_component(hint)\n        initial,_=principal_path(hint_binary,config.points)\n        if np.linalg.norm(initial[[0,-1]]-original[[0,-1]])>np.linalg.norm(initial[::-1][[0,-1]]-original[[0,-1]]):initial=initial[::-1]\n        if np.max(np.linalg.norm(initial-original,axis=1))>config.max_displacement:\n            initial=original.copy();metadata[\'warnings\'].append(\'Learned hint displacement too large; initialized from reference\')\n    except ValueError:\n        initial=original.copy();metadata[\'warnings\'].append(\'No usable learned hint; initialized from reference\')\n    target,target_info=principal_path(observed,config.points)\n    if np.linalg.norm(target[[0,-1]]-initial[[0,-1]])>np.linalg.norm(target[::-1][[0,-1]]-initial[[0,-1]]):target=target[::-1]\n    metadata[\'extracted_path\']=target_info\n    if target_info[\'closed\']:\n        target=initial.copy();metadata[\'warnings\'].append(\'Closed extracted path for open reference: retained initialization\')\n    if target_info[\'path_coverage\']<.75:metadata[\'warnings\'].append(\'Side branches excluded by principal-path fit\')\n    if component_info[\'selected_mass_fraction\']<.65:metadata[\'warnings\'].append(\'Large disconnected extraction mass excluded\')\n    chord=original[-1]-original[0];length=np.linalg.norm(np.diff(original,axis=0),axis=1).sum()\n    normal=np.array([-chord[1],chord[0]])/max(np.linalg.norm(chord),1e-6)\n    deviation=np.max(np.abs((original-original[0])@normal))\n    straight=deviation<=1.1 or deviation/max(length,1)<.025\n    if straight:\n        # Robust TLS: trim off-axis centerline points before estimating direction.\n        points=target.copy()\n        for _ in range(3):\n            center=points.mean(0);_,_,v=np.linalg.svd(points-center,full_matrices=False);axis=v[0]\n            residual=np.abs((target-center)@np.array([-axis[1],axis[0]]))\n            points=target[residual<=max(.75,float(np.quantile(residual,.85)))]\n        projection=(target-center)@axis\n        if axis@(initial[-1]-initial[0])<0:axis=-axis;projection=-projection\n        endpoints=np.quantile(projection,[.01,.99])\n        target=center+np.linspace(*endpoints,config.points)[:,None]*axis\n        metadata[\'method\']=\'robust_straight_reference_fit\'\n    else:metadata[\'method\']=\'regularized_curved_reference_fit\'\n    target_length=np.linalg.norm(np.diff(target,axis=0),axis=1).sum()\n    ratio=target_length/max(length,1e-6)\n    if not config.min_length_ratio<=ratio<=config.max_length_ratio:\n        metadata[\'warnings\'].append(\'Extracted path length outside allowed range; retained bounded initialization\')\n        target=initial.copy()\n        if straight:target=np.linspace(target[0],target[-1],config.points)\n    n=config.points;second=np.diff(np.eye(n),n=2,axis=0)\n    system=(1+config.shape_prior)*np.eye(n)+config.smoothness*(second.T@second)\n    if straight:\n        fitted=target\n    else:\n        # Smooth the deformation, rather than tracing every observed bend/spur.\n        fitted=initial+np.linalg.solve(system,target-initial)\n        endpoints=fitted[[0,-1]].copy()\n        fitted=gaussian_filter1d(fitted,.8,axis=0,mode=\'nearest\')\n        fitted[0],fitted[-1]=endpoints\n    displacement=fitted-original\n    norm=np.linalg.norm(displacement,axis=1,keepdims=True)\n    limit=min(1.,config.max_displacement/max(float(norm.max()),1e-6)) if straight else np.minimum(1,config.max_displacement/np.maximum(norm,1e-6))\n    fitted=original+displacement*limit\n    ref_width=_width(canonical,original)\n    target_width=float(np.clip(_width(observed,target),config.min_width,config.max_width))\n    width_scale=float(np.clip(target_width/ref_width,.5,8.))\n    warped=render_reference(reference,original,fitted,width_scale)\n    metadata.update(width_reference=ref_width,width_observed=target_width,width_scale=width_scale,\n        extracted_length_ratio=float(ratio),max_displacement=float(np.linalg.norm(fitted-original,axis=1).max()))\n    return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n        reference_path=original,target_path=fitted,hint_path=initial,width_scale=width_scale,metadata=metadata)\n\n\ndef fit_reference_glyph(strokes,extracted,*,hints=None,glyph_reference=None,config=None):\n    config=config or ReferenceFitConfig()\n    strokes=np.asarray(strokes,np.float32);extracted=np.asarray(extracted,np.float32)\n    if strokes.shape!=extracted.shape or strokes.ndim!=3:raise ValueError(\'K matching reference/extracted masks required\')\n    hints=strokes if hints is None else np.asarray(hints,np.float32)\n    if hints.shape!=strokes.shape:raise ValueError(\'Hint shape mismatch\')\n    fitted=[]\n    for k in range(len(strokes)):\n        try:item=fit_reference_stroke(strokes[k],extracted[k],hint=hints[k],config=config)\n        except ValueError as exc:\n            item=dict(warped=strokes[k].copy(),hint=hints[k],reference=strokes[k],extracted=extracted[k],\n                reference_path=None,target_path=None,hint_path=None,width_scale=1.,\n                metadata=dict(method=\'unreliable_reference_fallback\',warnings=[str(exc)]))\n        fitted.append(item)\n    # Joint bounded translations keep reference contacts from independently drifting.\n    matrix=np.eye(len(strokes));rhs=np.zeros((len(strokes),2));contacts=[]\n    for i,a in enumerate(fitted):\n        if a[\'reference_path\'] is None:continue\n        for j in range(i+1,len(fitted)):\n            b=fitted[j]\n            if b[\'reference_path\'] is None:continue\n            distance=np.linalg.norm(a[\'reference_path\'][:,None]-b[\'reference_path\'][None],axis=2)\n            u,v=np.unravel_index(distance.argmin(),distance.shape)\n            connected=distance[u,v]<=config.junction_radius\n            if not connected and glyph_reference is not None and distance[u,v]<=10:\n                line=np.linspace(a[\'reference_path\'][u],b[\'reference_path\'][v],32)\n                ink=map_coordinates(_unit(np.asarray(glyph_reference).squeeze()),[line[:,1],line[:,0]],order=1,mode=\'constant\')\n                connected=float((ink>.25).mean())>.85\n            if not connected:continue\n            # Sparse annotations often omit the segment inside a junction. If\n            # the full reference confirms a connection, extend an endpoint to\n            # the other stroke instead of translating its whole visible tip.\n            if glyph_reference is not None:\n                ia=0 if u<4 else -1 if u>=len(a[\'reference_path\'])-4 else None\n                ib=0 if v<4 else -1 if v>=len(b[\'reference_path\'])-4 else None\n                if (ia is None)!=(ib is None):\n                    end_item,end_index,other,other_index=(a,ia,b,v) if ia is not None else (b,ib,a,u)\n                    difference=other[\'target_path\'][other_index]-end_item[\'target_path\'][end_index]\n                    gap=float(np.linalg.norm(difference))\n                    if gap<=config.max_displacement:\n                        t=np.linspace(0,1,len(end_item[\'target_path\']))\n                        factor=(1-t)**3 if end_index==0 else t**3\n                        if end_item[\'metadata\'].get(\'method\')==\'robust_straight_reference_fit\':factor=(1-t) if end_index==0 else t\n                        end_item[\'target_path\']=end_item[\'target_path\']+factor[:,None]*difference\n                        end_item[\'metadata\'].setdefault(\'endpoint_extensions\',[]).append(gap)\n            difference=a[\'target_path\'][u]-b[\'target_path\'][v]\n            weight=config.junction_weight\n            matrix[i,i]+=weight;matrix[j,j]+=weight;matrix[i,j]-=weight;matrix[j,i]-=weight\n            rhs[i]-=weight*difference;rhs[j]+=weight*difference\n            contacts.append(dict(strokes=[i,j],indices=[int(u),int(v)],gap_before=float(np.linalg.norm(difference))))\n    shift=np.linalg.solve(matrix,rhs)\n    shift*=np.minimum(1,config.max_junction_shift/np.maximum(np.linalg.norm(shift,axis=1,keepdims=True),1e-6))\n    for i,item in enumerate(fitted):\n        if item[\'target_path\'] is None:continue\n        item[\'target_path\']=item[\'target_path\']+shift[i]\n        displacement=item[\'target_path\']-item[\'reference_path\']\n        maximum=float(np.linalg.norm(displacement,axis=1).max())\n        if maximum>config.max_displacement:\n            item[\'target_path\']=item[\'reference_path\']+displacement*(config.max_displacement/maximum)\n            item[\'metadata\'][\'warnings\'].append(\'Joint displacement clipped; inspect contact consistency\')\n        item[\'warped\']=render_reference(item[\'reference\'],item[\'reference_path\'],item[\'target_path\'],item[\'width_scale\'])\n        item[\'metadata\'][\'junction_translation\']=shift[i].tolist()\n        item[\'metadata\'][\'final_max_displacement\']=float(np.linalg.norm(item[\'target_path\']-item[\'reference_path\'],axis=1).max())\n    for contact in contacts:\n        i,j=contact[\'strokes\'];u,v=contact[\'indices\']\n        contact[\'gap_after\']=float(np.linalg.norm(fitted[i][\'target_path\'][u]-fitted[j][\'target_path\'][v]))\n    masks=np.stack([item[\'warped\'] for item in fitted])\n    return dict(strokes=masks,candidate=masks.max(0),fits=fitted,contacts=contacts,config=asdict(config))\n', 'diffusion_model/fitted_candidate_diffusion.py': '"""Paired DDIM re-denoising guided by an extractor-fitted candidate, not GT."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\n\n\ndef load_case(archive, case_id):\n    """Read named members only; recorded Colab absolute paths are not trusted."""\n    with zipfile.ZipFile(archive) as z:\n        if \'case_manifest.json\' not in z.namelist():\n            report = json.loads(z.read(\'report.json\'))\n            metadata = dict(report[\'metadata\'])\n            if metadata[\'case\'][\'case_id\'] != case_id:\n                raise ValueError(\'CASE_ID does not match the saved reverse experiment\')\n            with Image.open(io.BytesIO(z.read(\'input.png\'))) as im:\n                source = im.convert(\'RGB\').copy()\n            with Image.open(io.BytesIO(z.read(\'fitted_candidate.png\'))) as im:\n                target = im.convert(\'RGB\').copy()\n            if source.size != target.size or any(n % 8 for n in source.size):\n                raise ValueError(\'Input/candidate dimensions mismatch\')\n            metadata[\'reused_archive_kind\'] = \'reverse_result_original_input_and_candidate\'\n            return source, target, metadata\n        cases = json.loads(z.read(\'case_manifest.json\'))\n        matches = [c for c in cases if c[\'case_id\'] == case_id]\n        if len(matches) != 1:\n            raise ValueError(\'Select exactly one case from case_manifest.json\')\n        case = matches[0]\n        page = f"generation/page_{case[\'page_id\']:03d}.png"\n        candidate = f"{case_id}/normal_candidate.png"\n        def read_image(name):\n            with Image.open(io.BytesIO(z.read(name))) as im:\n                return im.convert(\'RGB\').copy()\n        source, target = read_image(page), read_image(candidate)\n        if source.size != target.size or any(n % 8 for n in source.size):\n            raise ValueError(\'Input and fitted candidate must have equal dimensions divisible by 8\')\n        report = json.loads(z.read(\'generation/generation_report.json\'))\n        metadata = dict(case=case, candidate_member=candidate, input_member=page,\n                        generation_model=report[\'model_id\'],\n                        candidate_is_ground_truth=False,\n                        prompt=report.get(\'prompts\', {}).get(\'4\', \'\'),\n                        negative_prompt=report.get(\'negative_prompt\', \'\'))\n    return source, target, metadata\n\n\ndef image_tensor(image, device):\n    a = np.asarray(image, dtype=np.float32) / 255\n    return torch.from_numpy(a.copy()).permute(2, 0, 1)[None].to(device)\n\n\nclass InkConstraintObjective:\n    """Candidate supplies fixed sparse constraints, never RGB target pixels."""\n    def __init__(self, source, candidate, *, enable_restore=False, tolerance=5.,\n                 halo=2, preserve_weight=.5, background_ink=.015, required_ink=.7):\n        from scipy.ndimage import distance_transform_edt, binary_dilation\n        from skimage.morphology import skeletonize\n        if source.shape != candidate.shape or source.shape[:2] != (1, 3):\n            raise ValueError(\'Expected matching single RGB sheets\')\n        if not np.isfinite(tolerance) or tolerance < 2 or type(halo) is not int or halo < 0:\n            raise ValueError(\'Invalid constraint tolerance/halo\')\n        if not 0 <= background_ink < required_ink <= 1 or preserve_weight < 0:\n            raise ValueError(\'Invalid ink constraints\')\n        self.source = source.detach()\n        self.preserve_weight = preserve_weight\n        self.background_ink, self.required_ink = background_ink, required_ink\n        source_ink = self.ink(source).detach().cpu().numpy()[0, 0]\n        target_ink = self.ink(candidate).detach().cpu().numpy()[0, 0]\n        source_binary, target_binary = source_ink > .35, target_ink > .35\n        if not source_binary.any() or not target_binary.any():\n            raise ValueError(\'Empty source or candidate: cannot infer safe correspondence\')\n        distance = distance_transform_edt(~target_binary)\n        # Ignore ordinary boundary/width differences within a tolerance corridor.\n        seeds = source_binary & (distance > tolerance)\n        nearby = binary_dilation(seeds, iterations=halo) if halo else seeds\n        erase = nearby*np.clip((distance-1)/max(tolerance-1, 1), 0, 1)\n        # Restore only an absent centerline, not the candidate\'s full silhouette.\n        missing = skeletonize(target_binary) & (distance_transform_edt(~source_binary) > tolerance)\n        restore = missing.astype(np.float32) if enable_restore else np.zeros_like(erase)\n        erase[restore > 0] = 0  # Never issue conflicting instructions.\n        def tensor(a):\n            return torch.as_tensor(a.copy(), dtype=torch.float32, device=source.device)[None, None]\n        self.erase, self.restore = tensor(erase), tensor(restore)\n        self.edit = torch.maximum(self.erase, self.restore)\n        occupied = (self.edit > 0).float()\n        self.preserve = 1-F.max_pool2d(occupied, 5, 1, 2)\n        self.metadata = dict(enable_restore=enable_restore, tolerance_pixels=tolerance,\n            halo_pixels=halo, preserve_weight=preserve_weight,\n            background_ink=background_ink, required_ink=required_ink,\n            erase_pixels=int((self.erase > 0).sum()), restore_pixels=int((self.restore > 0).sum()),\n            candidate_rgb_matching=False, masks_frozen_for_entire_pass=True)\n\n    @staticmethod\n    def ink(rgb):\n        return 1-(rgb*rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n\n    def project_gradient(self, gradient):\n        # A small latent context halo permits VAE reconstruction at edit boundaries.\n        support = F.adaptive_max_pool2d((self.edit > 0).float(), gradient.shape[-2:])\n        return gradient*F.max_pool2d(support, 3, 1, 1)\n\n    def __call__(self, image):\n        ink = self.ink(image)\n        remaining = F.relu(ink-self.background_ink)\n        erase = CandidateObjective.mean_region(remaining, self.erase)\n        # Penalize the darkest residual pixels as well as average remaining ink.\n        active = remaining[self.erase > 0]\n        tail = active.topk(max(1, int(active.numel()*.1))).values.mean() if active.numel() else erase*0\n        restore = CandidateObjective.mean_region(F.relu(self.required_ink-ink), self.restore)\n        preserve = CandidateObjective.mean_region((image-self.source).square(), self.preserve)\n        healthy_ink = self.preserve*(self.ink(self.source) > .2).float()\n        healthy_mse = CandidateObjective.mean_region((image-self.source).square(), healthy_ink)\n        loss = erase+.25*tail+restore+self.preserve_weight*preserve\n        return loss, dict(erase_ink=float(erase.detach()), erase_tail=float(tail.detach()),\n            restore_deficit=float(restore.detach()), preserved_region_mse=float(preserve.detach()),\n            healthy_ink_rgb_mse=float(healthy_mse.detach()))\n\n\nclass CandidateObjective:\n    """Fixed spatial weights; gradients stay continuous through the VAE."""\n    def __init__(self, source, candidate, threshold=.10, radius=2, preserve_weight=.25):\n        if source.shape != candidate.shape or not 0 < threshold < 1 or radius < 0:\n            raise ValueError(\'Invalid images or edit-region settings\')\n        self.source, self.candidate = source.detach(), candidate.detach()\n        difference = (source - candidate).abs().mean(1, keepdim=True)\n        edit = (difference > threshold).float()\n        self.edit = F.max_pool2d(edit, 2*radius+1, 1, radius).detach()\n        self.preserve_weight = preserve_weight\n\n    @staticmethod\n    def mean_region(values, mask):\n        return (values*mask).sum() / (mask.sum()*values.shape[1]).clamp_min(1)\n\n    def __call__(self, image):\n        match = self.mean_region((image-self.candidate).square(), self.edit)\n        preserve = self.mean_region((image-self.source).square(), 1-self.edit)\n        loss = match + self.preserve_weight*preserve\n        return loss, dict(candidate_region_mse=float(match.detach()),\n                          preserved_region_mse=float(preserve.detach()))\n\n\ndef optimize_clean(clean, decode, objective, *, iterations=4, step_ratio=.02,\n                   budget_ratio=.08, backtracks=5):\n    """Backtrack on the SAME clean estimate; bound total correction per step."""\n    anchor = clean.detach().float()\n    current = anchor.clone()\n    scale = anchor.square().mean().sqrt().clamp_min(1e-3)\n    with torch.no_grad():\n        before, _ = objective(decode(current))\n    accepted = 0\n    for _ in range(iterations):\n        with torch.enable_grad():\n            leaf = current.detach().requires_grad_(True)\n            loss, _ = objective(decode(leaf))\n            gradient, = torch.autograd.grad(loss, leaf)\n            if hasattr(objective, \'project_gradient\'):\n                gradient = objective.project_gradient(gradient)\n        if not torch.isfinite(gradient).all() or not torch.isfinite(loss):\n            raise FloatingPointError(\'Nonfinite candidate guidance\')\n        norm = gradient.square().mean().sqrt()\n        if norm < 1e-12:\n            break\n        direction = step_ratio*scale*gradient/norm\n        found = False\n        with torch.no_grad():\n            for attempt in range(backtracks+1):\n                proposal = current-direction*(.5**attempt)\n                delta = proposal-anchor\n                rms = delta.square().mean().sqrt().clamp_min(1e-12)\n                proposal = anchor+delta*(budget_ratio*scale/rms).clamp(max=1)\n                trial, _ = objective(decode(proposal))\n                if torch.isfinite(trial) and trial < loss.detach():\n                    current = proposal.detach()\n                    accepted += 1\n                    found = True\n                    break\n        if not found:\n            break\n    with torch.no_grad():\n        after, details = objective(decode(current))\n    return current, dict(loss_before=float(before), loss_after=float(after),\n                         accepted_inner_steps=accepted, **details)\n\n\ndef run_pair(source, candidate, metadata, output, *, steps=60, strength=.35,\n             seed=2026, cfg_scale=5., inner_steps=4, step_ratio=.02,\n             model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\', revision=None,\n             progress=None, guidance_mode=\'candidate\', constraint_config=None, objective_override=None):\n    """Identical source VAE encoding, noise, prompt, scheduler in both arms."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select an L4 GPU runtime in Colab\')\n    if not 0 < strength <= 1 or steps < 4 or int(steps*strength) < 2:\n        raise ValueError(\'Need at least two actual re-denoising steps\')\n    if inner_steps < 1 or step_ratio <= 0 or cfg_scale <= 1:\n        raise ValueError(\'Invalid guidance configuration\')\n    if guidance_mode not in (\'candidate\', \'ink_constraints\', \'adaptive_ink\'):\n        raise ValueError(\'Unknown guidance mode\')\n    if (guidance_mode == \'adaptive_ink\') != (objective_override is not None):\n        raise ValueError(\'Adaptive mode requires an explicitly prepared stroke objective\')\n    from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n    from copy import deepcopy\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    source.save(out/\'input.png\')\n    candidate.save(out/\'fitted_candidate.png\')\n    kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n    if revision:\n        kwargs[\'revision\'] = revision\n    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config,\n                                              clip_sample=False, thresholding=False)\n    pipe.to(\'cuda\')\n    pipe.enable_attention_slicing()\n    pipe.enable_vae_slicing()\n    for module in (pipe.unet, pipe.vae, pipe.text_encoder):\n        module.eval().requires_grad_(False)\n    prompt = metadata.get(\'prompt\') or (\'four Korean Hangul glyphs, black ink on white, \'\n                                      \'preserve the input typography and layout\')\n    with torch.no_grad():\n        positive, negative = pipe.encode_prompt(prompt, \'cuda\', 1, True,\n            negative_prompt=metadata.get(\'negative_prompt\', \'\'))\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        # VAE and noise initialization exactly once, then copy for both arms.\n        initial = pipe.prepare_latents(pipe.image_processor.preprocess(source), timesteps[:1],\n            1, 1, embeddings.dtype, \'cuda\', torch.Generator(\'cpu\').manual_seed(seed))\n    pipe.vae.to(dtype=torch.float32)\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor,\n                               return_dict=False)[0].float()/2+.5\n    def save_image(z, path):\n        with torch.no_grad():\n            array = decode(z)[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(array*255).astype(\'uint8\')).save(path)\n    source_rgb, candidate_rgb = image_tensor(source, \'cuda\'), image_tensor(candidate, \'cuda\')\n    if objective_override is not None and (objective_override.source.shape != source_rgb.shape or\n            not torch.allclose(objective_override.source, source_rgb, atol=1e-6)):\n        raise ValueError(\'Adaptive objective must be prepared from this exact original input\')\n    objective = objective_override if objective_override is not None else (InkConstraintObjective(source_rgb, candidate_rgb, **(constraint_config or {}))\n                 if guidance_mode == \'ink_constraints\' else CandidateObjective(source_rgb, candidate_rgb))\n    Image.fromarray((objective.edit[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/\'edit_region.png\')\n    if guidance_mode in (\'ink_constraints\', \'adaptive_ink\'):\n        for name in (\'erase\', \'restore\', \'preserve\'):\n            Image.fromarray(np.rint(getattr(objective, name)[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/f\'{name}_region.png\')\n        (out/\'constraints.json\').write_text(json.dumps(objective.metadata, indent=2), encoding=\'utf-8\')\n    guided_name = {\'candidate\': \'candidate_guided\', \'ink_constraints\': \'ink_guided\', \'adaptive_ink\': \'adaptive_guided\'}[guidance_mode]\n    if hasattr(objective, \'export_state\'):\n        objective.export_state(out/\'ownership_initial.npz\')\n    traces, metrics = {}, {}\n    for method in (\'redenoise_only\', guided_name):\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.detach().float().clone()\n        trace = []\n        for i, t in enumerate(timesteps):\n            with torch.no_grad():\n                inputs = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(inputs.to(torch.float16), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                noise = (u+cfg_scale*(c-u)).float()\n                base = scheduler.step(noise, t, latent, eta=0., return_dict=True)\n            record = dict(step=i, timestep=int(t), active=False, accepted_inner_steps=0)\n            latent = base.prev_sample.detach()\n            clean = base.pred_original_sample.detach()\n            if method == guided_name and i/max(len(timesteps)-1, 1) >= .2:\n                if hasattr(objective, \'maybe_refresh\'):\n                    with torch.no_grad():\n                        refresh = objective.maybe_refresh(decode(clean), i)\n                    if refresh is not None:\n                        record[\'constraint_refresh\'] = refresh\n                        for name in (\'erase\', \'restore\'):\n                            mask = getattr(objective, name)[0, 0].detach().cpu().numpy()\n                            Image.fromarray(np.rint(mask*255).clip(0, 255).astype(\'uint8\')).save(out/f\'{name}_step_{i:03d}.png\')\n                        if hasattr(objective, \'export_state\'):\n                            objective.export_state(out/f\'ownership_step_{i:03d}.npz\')\n                clean, result = optimize_clean(clean, decode, objective,\n                    iterations=inner_steps if i < len(timesteps)-1 else inner_steps*2,\n                    step_ratio=step_ratio)\n                # DDIM eta=0: x_(t-1) = sqrt(a_prev)*x0 + sqrt(1-a_prev)*epsilon.\n                # Hold the predicted epsilon fixed and change the clean component.\n                prev_t = int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[prev_t] if prev_t >= 0\n                         else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent+alpha.sqrt()*(clean-base.pred_original_sample)\n                record.update(active=True, **result)\n            latent = latent.detach()\n            trace.append(record)\n            if i % 4 == 0 or i == len(timesteps)-1:\n                save_image(clean, out/f\'{method}_step_{i:03d}.png\')\n            if progress:\n                progress(method, i+1, len(timesteps), record)\n        save_image(latent, out/f\'{method}.png\')\n        with torch.no_grad():\n            evaluate = getattr(objective, \'evaluate_fixed\', objective)\n            loss, values = evaluate(decode(latent))\n        metrics[method] = dict(loss=float(loss), **values)\n        traces[method] = trace\n    with torch.no_grad():\n        loss, values = getattr(objective, \'evaluate_fixed\', objective)(objective.source)\n    metrics[\'input\'] = dict(loss=float(loss), **values)\n    report = dict(metadata=metadata, model_id=model_id, revision=revision,\n        steps=steps, actual_steps=len(timesteps), strength=strength, seed=seed,\n        cfg_scale=cfg_scale, inner_steps=inner_steps, step_ratio=step_ratio,\n        initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n        paired_initial_latent=True, guidance_target=guidance_mode,\n        constraints=getattr(objective, \'metadata\', None),\n        constraint_history=getattr(objective, \'history\', None),\n        target_pixels_pasted_into_output=False, metrics=metrics, traces=traces,\n        metric_note=(\'Fixed ink constraints and source preservation; not GT accuracy or HSQM score\'\n                     if guidance_mode in (\'ink_constraints\', \'adaptive_ink\') else \'Candidate agreement, not ground-truth accuracy or HSQM score\'))\n    (out/\'report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    del pipe\n    torch.cuda.empty_cache()\n    return report\n', 'stroke_model/restore_settle_erase.py': '"""Reference-path insertion, settling, then re-extracted local erasure.\n\nOnly missing structural ink is inserted in clean-image space. VAE round-trip\ncoverage is checked; unresolved restoration never enables the erasure phase.\n"""\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom scipy.ndimage import affine_transform, distance_transform_edt, binary_dilation, convolve, label\nfrom skimage.morphology import skeletonize\n\nfrom diffusion_model.fitted_candidate_diffusion import InkConstraintObjective, CandidateObjective\n\n\ndef centroid(mask):\n    yy, xx = np.indices(mask.shape)\n    mass = max(float(mask.sum()), 1e-6)\n    return np.array([(xx*mask).sum()/mass, (yy*mask).sum()/mass])\n\n\ndef align_prototypes(record):\n    """Visible fitted anchors position missing prototypes; absent ink is not fit."""\n    source, target = [], []\n    masks = record.get(\'extracted_masks\', record[\'fitted\'])\n    metadata = record.get(\'fit_metadata\', [{} for _ in masks])\n    for original, fitted, observed, info in zip(record[\'strokes\'], record[\'fitted\'], masks, metadata):\n        if float(observed.max()) < .3 or info.get(\'warnings\') or \'fallback\' in info.get(\'method\', \'\'):continue\n        source.append(centroid(original));target.append(centroid(fitted))\n    matrix = np.c_[np.eye(2), np.zeros(2)]\n    kind = \'canonical_fallback_no_reliable_visible_anchors\'\n    if source:\n        x, y = np.array(source), np.array(target)\n        shift = np.median(y-x, axis=0).clip(-12, 12)\n        matrix[:, 2] = shift;kind = \'visible_anchor_translation\'\n        if len(x) >= 3 and np.linalg.matrix_rank(np.c_[x, np.ones(len(x))]) == 3:\n            fitted, *_ = np.linalg.lstsq(np.c_[x, np.ones(len(x))], y, rcond=None)\n            candidate = fitted.T\n            singular = np.linalg.svd(candidate[:, :2], compute_uv=False)\n            residual = np.linalg.norm(np.c_[x, np.ones(len(x))]@fitted-y, axis=1)\n            corners=np.array([[0,0],[0,record[\'strokes\'].shape[-2]-1],\n                              [record[\'strokes\'].shape[-1]-1,0],\n                              [record[\'strokes\'].shape[-1]-1,record[\'strokes\'].shape[-2]-1]])\n            movement=np.linalg.norm(np.c_[corners,np.ones(4)]@fitted-corners,axis=1).max()\n            if singular.min() >= .8 and singular.max() <= 1.25 and np.median(residual) < 3 and movement <= 16:\n                matrix = candidate;kind = \'bounded_visible_anchor_affine\'\n    swap = np.array([[0, 1], [1, 0]])\n    inverse = np.linalg.inv(matrix[:, :2])\n    warped = np.stack([affine_transform(s, swap@inverse@swap, offset=-swap@inverse@matrix[:, 2],\n                      output_shape=s.shape, order=1, mode=\'constant\') for s in record[\'strokes\']])\n    return warped.clip(0, 1), dict(method=kind, matrix=matrix.tolist(), visible_anchors=len(source))\n\n\nclass RestoreSettleErase(InkConstraintObjective):\n    def __init__(self, source, extractor, records, *, coverage_threshold=.85,\n                 max_cycles=2, settle_steps=2):\n        if len(records) != 4 or source.shape[:2] != (1, 3):raise ValueError(\'Four slots and one RGB sheet required\')\n        if not .5 < coverage_threshold <= 1 or max_cycles < 1 or settle_steps < 1:raise ValueError(\'Invalid cycle schedule\')\n        self.source = source.detach();self.extractor = extractor.eval().requires_grad_(False)\n        self.threshold = coverage_threshold;self.max_cycles = max_cycles;self.settle_steps = settle_steps\n        self.phase = \'restore\';self.cycles = 1;self.settle_left = settle_steps\n        self.background_ink = .015;self.required_ink = .7;self.preserve_weight = .5\n        self.records = [];self.coverage_log = [];self.history = []\n        self.erase = torch.zeros_like(self.ink(source));self.restore = torch.zeros_like(self.erase)\n        self.required_paths = [];self.initial_missing = []\n        h, w = source.shape[-2:]\n        boxes = [(0,0,w//2,h//2),(w//2,0,w,h//2),(0,h//2,w//2,h),(w//2,h//2,w,h)]\n        for slot, (raw, box) in enumerate(zip(records, boxes)):\n            prototypes, alignment = align_prototypes(raw)\n            size = raw[\'strokes\'].shape[-2:]\n            observed = self.crop(self.ink(source), box, size)[0, 0].cpu().numpy()\n            proto_paths = [skeletonize(p > .2) for p in prototypes]\n            visible_widths = []\n            for mask, path in zip(raw[\'fitted\'], proto_paths):\n                values = distance_transform_edt(mask > .35)[path & (observed > .35)]\n                if values.size >= 4:visible_widths.append(float(np.median(values)))\n            borrowed_half = float(np.clip(np.median(visible_widths) if visible_widths else 2., 1.25, 5.))\n            distances = [];radii = [];identities = []\n            for k, (prototype_path, fitted) in enumerate(zip(proto_paths, raw[\'fitted\'])):\n                if not prototype_path.any():raise ValueError(f\'Slot {slot+1} stroke {k+1}: empty prototype\')\n                prototype_coverage = float((observed[prototype_path] > .35).mean())\n                info = raw.get(\'fit_metadata\', [{} for _ in proto_paths])[k]\n                unreliable = prototype_coverage < .35 or bool(info.get(\'warnings\')) or \'fallback\' in info.get(\'method\', \'\')\n                path = prototype_path if unreliable else skeletonize(fitted > .35)\n                if not path.any():path = prototype_path\n                distance = distance_transform_edt(~path)\n                # Present strokes keep their original caps/serifs. Small endpoint\n                # offsets are not missing structure. Whole absent strokes retain caps.\n                if prototype_coverage >= .2:\n                    degree=convolve(path.astype(np.int32),np.ones((3,3),np.int32),mode=\'constant\')-path\n                    ends=path&(degree==1)\n                    if ends.any():\n                        interior=path&(distance_transform_edt(~ends)>2.)\n                        if interior.any():path=interior\n                values = distance_transform_edt(fitted > .35)[path & (observed > .35)]\n                half = float(np.clip(np.median(values) if values.size >= 4 else borrowed_half, 1.25, 5.))\n                # A narrow, renderable band supplies minimum structure, not a full silhouette.\n                missing = path & (distance_transform_edt(~(observed > .3)) > .75)\n                if prototype_coverage >= .2:\n                    components,count=label(missing,np.ones((3,3)))\n                    minimum=max(2,int(np.ceil(path.sum()*.03)))\n                    missing=np.isin(components,[j for j in range(1,count+1) if (components==j).sum()>=minimum])\n                scaffold = binary_dilation(missing, iterations=max(1, int(round(half*.4))))\n                canvas = torch.zeros_like(self.restore);self.place(canvas, self.tensor(scaffold), box)\n                self.restore = torch.maximum(self.restore, canvas)\n                core = torch.zeros_like(self.restore);self.place(core, self.tensor(path), box)\n                core = (core > .2).float()\n                self.required_paths.append(core)\n                self.initial_missing.append(dict(slot=slot+1, stroke=k+1,\n                    coverage=prototype_coverage, missing_path_pixels=int(missing.sum()),\n                    entire_stroke_missing=prototype_coverage < .2, prototype_fallback=unreliable))\n                distances.append(distance);radii.append(half);identities.append(k+1)\n            self.records.append(dict(box=box,size=size,reference=self.tensor(raw[\'reference\']).reshape(1,1,*size),\n                strokes=torch.as_tensor(raw[\'strokes\'],device=source.device,dtype=torch.float32)[None],\n                distances=np.stack(distances),radii=np.asarray(radii)[:,None,None],\n                alignment=alignment,stroke_ids=identities))\n        self.restore = (self.restore > .2).float()\n        self.path_union = torch.stack(self.required_paths).amax(0)\n        self.initial_restore = self.restore.clone()\n        self.update_support()\n        self.fixed_preserve = self.preserve.clone()\n        self.metadata = dict(kind=\'restore_settle_erase\', max_cycles=max_cycles, settle_steps=settle_steps,\n            coverage_threshold=coverage_threshold, missing_strokes=self.initial_missing,\n            alignments=[r[\'alignment\'] for r in self.records], candidate_rgb_matching=False,\n            structural_ink_projection=True, restore_before_erase=True)\n\n    def tensor(self, array):return torch.as_tensor(np.asarray(array).copy(),device=self.source.device,dtype=torch.float32)[None,None]\n    @staticmethod\n    def crop(image, box, size):\n        x0,y0,x1,y1=box\n        return F.interpolate(image[...,y0:y1,x0:x1],size,mode=\'bilinear\',align_corners=False)\n    @staticmethod\n    def place(canvas, patch, box):\n        x0,y0,x1,y1=box\n        canvas[...,y0:y1,x0:x1]=F.interpolate(patch,(y1-y0,x1-x0),mode=\'bilinear\',align_corners=False)\n\n    def update_support(self):\n        self.edit=torch.maximum(self.erase,self.restore)\n        self.preserve=1-F.max_pool2d((self.edit>.01).float(),5,1,2)\n\n    @torch.no_grad()\n    def coverage(self, image):\n        ink=self.ink(image)\n        return [float(((ink>.5).float()*p).sum()/p.sum().clamp_min(1)) for p in self.required_paths]\n\n    @torch.no_grad()\n    def insertion_image(self, image):\n        """Explicit minimum-ink projection inside the missing band only."""\n        deficiency=F.relu(self.required_ink-self.ink(image))*(self.restore>.01).float()\n        return (image-deficiency).clamp(0,1)\n\n    @torch.no_grad()\n    def project_clean(self, clean, decode, encode, rounds=3):\n        current=clean.detach().float();before=self.coverage(decode(current))\n        support=F.adaptive_max_pool2d((self.restore>.01).float(),clean.shape[-2:])\n        support=F.max_pool2d(support,3,1,1)\n        attempts=0\n        for _ in range(rounds):\n            image=decode(current)\n            if min(self.coverage(image))>=self.threshold:break\n            # A path newly lost after deletion is also eligible for restoration.\n            target=self.insertion_image(image)\n            projected=encode(target).detach().float()\n            if not torch.isfinite(projected).all():raise FloatingPointError(\'Nonfinite structural projection\')\n            current=current+support*(projected-current)\n            attempts+=1\n        after=self.coverage(decode(current))\n        return current,dict(projection_rounds=attempts,coverage_before=before,coverage_after=after,\n            restoration_verified=min(after)>=self.threshold)\n\n    @torch.no_grad()\n    def refresh_erase(self, image):\n        erase=torch.zeros_like(self.erase)\n        for r in self.records:\n            ink=self.crop(self.ink(image).clamp(0,1),r[\'box\'],r[\'size\'])\n            p=self.extractor(ink,r[\'reference\'],r[\'strokes\'])[\'probabilities\'][0].float().cpu().numpy()\n            if not np.isfinite(p).all() or p.shape!=r[\'distances\'].shape:raise ValueError(\'Invalid stroke extraction\')\n            # Protect complete expected stroke corridors, including newly inserted paths.\n            outside=np.clip((r[\'distances\']-r[\'radii\']-1.5)/1.5,0,1)\n            safe=(r[\'distances\']<=r[\'radii\']+1.).any(0)\n            responsibility=(p*outside).sum(0)\n            observed=ink[0,0].cpu().numpy()\n            seeds=(observed>.15)&(responsibility>.5)&~safe\n            weight=binary_dilation(seeds,iterations=1)*responsibility*(~safe)\n            self.place(erase,self.tensor(weight),r[\'box\'])\n        self.erase=erase*(self.restore<=0).float()*(1-self.path_union)\n        self.update_support()\n\n    @torch.no_grad()\n    def begin_step(self, image, index):\n        cov=self.coverage(image)\n        if min(cov)<self.threshold and self.phase==\'restore\':\n            missing=self.path_union*(self.ink(image)<.5).float()\n            self.restore=torch.maximum(self.restore,F.max_pool2d(missing,3,1,1))\n        if min(cov)<self.threshold and self.phase!=\'restore\':\n            if self.cycles<self.max_cycles:\n                self.cycles+=1;self.phase=\'restore\'\n                # Enlarge the restoration band only around paths that were lost.\n                missing=self.path_union*(self.ink(image)<.5).float()\n                self.restore=torch.maximum(self.restore,F.max_pool2d(missing,3,1,1))\n            else:self.phase=\'unresolved\'\n        if self.phase!=\'erase\':self.erase.zero_();self.update_support()\n        elif index%2==0:self.refresh_erase(image)\n        return dict(step=index,phase=self.phase,cycle=self.cycles,coverage_before=cov)\n\n    def finish_step(self, image, row):\n        cov=self.coverage(image);verified=min(cov)>=self.threshold\n        if self.phase==\'restore\' and verified:self.phase=\'settle\';self.settle_left=self.settle_steps\n        elif self.phase==\'settle\':\n            self.settle_left-=1\n            if self.settle_left<=0 and verified:\n                self.phase=\'erase\';self.refresh_erase(image)\n        row.update(coverage_after=cov,restoration_verified=verified,next_phase=self.phase,\n                   erase_pixels=int((self.erase>.01).sum()))\n        self.history.append(row)\n\n    def __call__(self, image):\n        loss,details=super().__call__(image)\n        deficits=[CandidateObjective.mean_region(F.relu(.6-self.ink(image)),p) for p in self.required_paths]\n        path_loss=torch.stack(deficits).mean()\n        details[\'path_deficit\']=float(path_loss.detach())\n        return loss+path_loss,details\n\n    def diagnostics(self, image):\n        cov=self.coverage(image)\n        keep=CandidateObjective.mean_region((image-self.source).square(),self.fixed_preserve)\n        return dict(per_stroke_coverage=cov,min_coverage=min(cov),normal_region_mse=float(keep),\n                    all_paths_present=min(cov)>=self.threshold)\n', 'repair_audit/stage1.py': '"""First-stage diagnosis of the existing four-glyph cyclic repair, without training.\n\nReads data (never runtime code) from existing sweep/cyclic ZIPs. Automatic width\nand reference-distance measurements are proxies, not character correctness.\n"""\nfrom copy import deepcopy\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.ndimage import binary_dilation, distance_transform_edt, maximum_filter\nfrom skimage.morphology import skeletonize\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef json_write(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef _rgb(z, name):\n    with Image.open(io.BytesIO(z.read(name))) as image:\n        return image.convert(\'RGB\').copy()\n\n\ndef _conditions(z, group):\n    result = []\n    for slot in range(1, 5):\n        member = f\'{group}/source_conditions/condition_{slot}.npz\'\n        with np.load(io.BytesIO(z.read(member)), allow_pickle=False) as arrays:\n            reference = arrays[\'reference\'].astype(np.float32)\n            strokes = arrays[\'strokes\'].astype(np.float32)\n        if (reference.ndim != 3 or reference.shape[0] != 1 or\n                reference.shape[1] != reference.shape[2] or strokes.ndim != 3 or\n                strokes.shape[1:] != reference.shape[1:] or not len(strokes)):\n            raise ValueError(f\'Invalid reference/stroke dimensions: {member}\')\n        for value in (reference, strokes):\n            if not np.isfinite(value).all() or value.min() < 0 or value.max() > 1:\n                raise ValueError(f\'Expected finite [0,1] ink arrays: {member}\')\n        if reference.max() <= .5 or (strokes.max((1, 2)) <= .5).any():\n            raise ValueError(f\'Empty glyph/stroke: {member}\')\n        result.append(dict(reference=reference, strokes=strokes))\n    return result\n\n\ndef read_archive(path, output, selected_case_ids=None, include_controls=True):\n    """Materialize only named PNG/NPZ data, ignoring saved absolute paths/code."""\n    with zipfile.ZipFile(path) as check:\n        if \'archive_metadata.json\' in check.namelist():\n            return read_stage1_archive(path, output, selected_case_ids, include_controls)\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    selected = None if selected_case_ids is None else set(selected_case_ids)\n    with zipfile.ZipFile(path) as z:\n        experiment = json.loads(z.read(\'experiment.json\'))\n        expected = experiment.get(\'checkpoint_sha256\', \'\')\n        if not re.fullmatch(r\'[0-9a-fA-F]{64}\', expected):\n            raise ValueError(\'Archive experiment.json must record the extractor SHA256\')\n        cases = json.loads(z.read(\'cases.json\'))\n        available = {c[\'case_id\'] for c in cases if not c.get(\'control\')}\n        if selected is not None and (not selected or selected - available):\n            raise ValueError(f\'Unknown/empty case selection; available: {sorted(available)}\')\n        chosen = [c for c in cases if not c.get(\'control\') and\n                  (selected is None or c[\'case_id\'] in selected)]\n        if not chosen:\n            raise ValueError(\'No generated cases in this archive\')\n        prepared, controls = [], set()\n        destination.mkdir(parents=True)\n        for item in chosen:\n            group, case_id = item[\'group_id\'], item[\'case_id\']\n            if not re.fullmatch(r\'group_[0-9]+\', group) or not re.fullmatch(r\'[A-Za-z0-9_-]+\', case_id):\n                raise ValueError(\'Invalid archive group/case identifier\')\n            conditions = _conditions(z, group)\n            report = json.loads(z.read(f\'{group}/generation/generation_report.json\'))\n            basename = PurePosixPath(str(item[\'input\']).replace(\'\\\\\', \'/\')).name\n            matches = [p for p in report[\'pages\'] if\n                       PurePosixPath(p[\'input\']).name == basename and not p.get(\'control\')]\n            if len(matches) != 1:\n                raise ValueError(f\'Generated page is not uniquely recorded: {case_id}\')\n            pages = [(case_id, matches[0], \'generated\')]\n            if include_controls and group not in controls:\n                candidates = [p for p in report[\'pages\'] if p.get(\'control\') and p[\'glyph_count\'] == 4]\n                if len(candidates) != 1:\n                    raise ValueError(\'Need one recorded four-glyph normal input control\')\n                pages.append((group + \'_normal_control\', candidates[0], \'normal_control\'))\n                controls.add(group)\n            for identifier, page, kind in pages:\n                if page[\'glyph_count\'] != 4 or page.get(\'safety_filtered\'):\n                    raise ValueError(\'Requires a non-filtered four-glyph page\')\n                member = f"{group}/generation/page_{page[\'page_id\']:03d}.png"\n                source = _rgb(z, member)\n                if source.width != source.height or source.width % 16:\n                    raise ValueError(\'Expected square four-cell image with side divisible by 16\')\n                target = destination / identifier\n                target.mkdir()\n                source.save(target / \'input.png\')\n                for slot, condition in enumerate(conditions, 1):\n                    np.savez_compressed(target / f\'condition_{slot}.npz\', **condition)\n                metadata = dict(case_id=identifier, kind=kind, archive_member=member,\n                    source_sha256=hashlib.sha256(z.read(member)).hexdigest(),\n                    prompt=report.get(\'prompts\', {}).get(\'4\', item.get(\'prompt\', \'\')),\n                    negative_prompt=report.get(\'negative_prompt\', \'\'),\n                    generation_seed=page.get(\'seed\'), generation_strength=page.get(\'strength\'),\n                    checkpoint_sha256=expected.lower())\n                json_write(target / \'input_metadata.json\', metadata)\n                prepared.append(dict(directory=str(target), **metadata))\n    json_write(destination / \'archive_metadata.json\', dict(archive_sha256=file_hash(path),\n        experiment=experiment, selected_cases=prepared))\n    return prepared, experiment\n\n\ndef read_stage1_archive(path, output, selected_case_ids=None, include_controls=True):\n    """Reuse original inputs from an audit ZIP, never the already guided outputs."""\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    with zipfile.ZipFile(path) as z:\n        metadata = json.loads(z.read(\'archive_metadata.json\'))\n        experiment = metadata[\'experiment\']\n        expected = experiment.get(\'checkpoint_sha256\', \'\')\n        if not re.fullmatch(r\'[0-9a-fA-F]{64}\', expected):\n            raise ValueError(\'Missing original extractor SHA256\')\n        recorded = metadata[\'selected_cases\']\n        available = {c[\'case_id\'] for c in recorded if c[\'kind\'] == \'generated\'}\n        selected = available if selected_case_ids is None else set(selected_case_ids)\n        if not selected or selected - available:\n            raise ValueError(f\'Unknown/empty audit selection; available: {sorted(available)}\')\n        chosen = [c for c in recorded if c[\'case_id\'] in selected]\n        groups = {re.fullmatch(r\'(group_[0-9]+)_page_[0-9]+\', c[\'case_id\']).group(1)\n                  for c in chosen if re.fullmatch(r\'(group_[0-9]+)_page_[0-9]+\', c[\'case_id\'])}\n        if len(groups) == 0:\n            raise ValueError(\'Unsupported audit case identifiers\')\n        if include_controls:\n            controls = {c[\'case_id\']: c for c in recorded if c[\'kind\'] == \'normal_control\'}\n            for group in sorted(groups):\n                name = group + \'_normal_control\'\n                if name not in controls:\n                    raise ValueError(\'Audit ZIP lacks the matching normal control\')\n                chosen.append(controls[name])\n        prepared = []\n        destination.mkdir(parents=True)\n        for entry in chosen:\n            identifier = entry[\'case_id\']\n            if not re.fullmatch(r\'group_[0-9]+_(page_[0-9]+|normal_control)\', identifier):\n                raise ValueError(\'Invalid audit identifier\')\n            target = destination / identifier\n            target.mkdir()\n            image = _rgb(z, identifier + \'/input.png\')\n            if image.width != image.height or image.width % 16:\n                raise ValueError(\'Invalid input sheet dimensions\')\n            image.save(target / \'input.png\')\n            # Same validation as the original four source conditions.\n            for slot in range(1, 5):\n                with np.load(io.BytesIO(z.read(f\'{identifier}/condition_{slot}.npz\')), allow_pickle=False) as a:\n                    ref, strokes = a[\'reference\'].astype(np.float32), a[\'strokes\'].astype(np.float32)\n                if (ref.ndim != 3 or ref.shape[0] != 1 or ref.shape[1] != ref.shape[2] or\n                        strokes.ndim != 3 or not len(strokes) or strokes.shape[1:] != ref.shape[1:]):\n                    raise ValueError(\'Invalid condition shape\')\n                if any(not np.isfinite(v).all() or v.min() < 0 or v.max() > 1 for v in (ref, strokes)):\n                    raise ValueError(\'Invalid condition values\')\n                if ref.max() <= .5 or (strokes.max((1, 2)) <= .5).any():\n                    raise ValueError(\'Empty condition\')\n                np.savez_compressed(target/f\'condition_{slot}.npz\', reference=ref, strokes=strokes)\n            row = dict(entry, directory=str(target), checkpoint_sha256=expected.lower(),\n                       reused_stage1_original=True)\n            json_write(target/\'input_metadata.json\', row)\n            prepared.append(row)\n    json_write(destination/\'archive_metadata.json\', dict(archive_sha256=file_hash(path),\n        experiment=experiment, selected_cases=prepared, original_input_reused=True))\n    return prepared, experiment\n\n\ndef load_conditions(directory):\n    result = []\n    for slot in range(1, 5):\n        with np.load(Path(directory) / f\'condition_{slot}.npz\', allow_pickle=False) as a:\n            result.append(dict(reference=a[\'reference\'].copy(), strokes=a[\'strokes\'].copy()))\n    return result\n\n\ndef fit_records(source, conditions, extractor, output, *, fit=True):\n    """Same fitting parameters as the existing cyclic notebook."""\n    import torch\n    from torch.nn import functional as F\n    from stroke_model.stroke_reference_fit import fit_reference_glyph, ReferenceFitConfig\n    from diffusion_model.fitted_candidate_diffusion import image_tensor\n    directory = Path(output)\n    directory.mkdir(parents=True, exist_ok=True)\n    device = next(extractor.parameters()).device\n    rgb = image_tensor(source, device)\n    ink = 1 - (rgb * rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n    side = source.width // 2\n    records = []\n    for i, condition in enumerate(conditions):\n        ref, strokes = condition[\'reference\'], condition[\'strokes\']\n        x, y = (i % 2) * side, (i // 2) * side\n        crop = F.interpolate(ink[..., y:y+side, x:x+side], ref.shape[-2:], mode=\'bilinear\', align_corners=False)\n        with torch.no_grad():\n            prediction = extractor(crop, torch.as_tensor(ref, device=device)[None],\n                                   torch.as_tensor(strokes, device=device)[None])\n        masks = prediction[\'masks\'][0].cpu().numpy()\n        if not fit:\n            np.savez_compressed(directory / f\'slot_{i+1}.npz\', **condition,\n                extracted_masks=masks, probabilities=prediction[\'probabilities\'][0].cpu().numpy())\n            continue\n        hints = prediction.get(\'hints\', torch.as_tensor(strokes, device=device)[None])[0].cpu().numpy()\n        fitted = fit_reference_glyph(strokes, masks, hints=hints, glyph_reference=ref,\n            config=ReferenceFitConfig(max_displacement=16., smoothness=12., shape_prior=.15, max_junction_shift=2.))\n        record = dict(reference=ref, strokes=strokes, fitted=fitted[\'strokes\'], extracted_masks=masks,\n                      fit_metadata=[r[\'metadata\'] for r in fitted[\'fits\']])\n        records.append(record)\n        np.savez_compressed(directory / f\'slot_{i+1}.npz\', **condition,\n            extracted_masks=masks, fitted_strokes=fitted[\'strokes\'],\n            probabilities=prediction[\'probabilities\'][0].cpu().numpy())\n        json_write(directory / f\'slot_{i+1}_fit.json\', record[\'fit_metadata\'])\n    return records\n\n\ndef ink_array(image):\n    a = np.asarray(image, np.float32) / 255\n    return 1 - (a * np.array([.299, .587, .114], np.float32)).sum(-1)\n\n\ndef fixed_support(source, controller):\n    """Freeze source-based diagnostic support before comparing ANY output arm."""\n    initial_edit = np.maximum(controller.erase.cpu().numpy()[0, 0], controller.restore.cpu().numpy()[0, 0]) > .01\n    preserve = ~binary_dilation(initial_edit, iterations=3)\n    source_ink = ink_array(source)\n    healthy = (source_ink > .5) & preserve\n    center = skeletonize(source_ink > .5) & healthy\n    return dict(healthy=healthy, center=center, preserve=preserve, initial_edit=initial_edit)\n\n\ndef measure(source, result, reference, support, threshold=.5):\n    """Fixed-coordinate width proxy; movement/thresholds confound it. No OCR claim."""\n    original, output, ref = map(ink_array, (source, result, reference))\n    healthy, center = support[\'healthy\'], support[\'center\']\n    source_radius = distance_transform_edt(original > threshold)[center]\n    output_radius = distance_transform_edt(output > threshold)[center]\n    ratio = float(output_radius.mean() / max(source_radius.mean(), 1e-8)) if len(source_radius) else None\n    union = (original > .2) | (ref > .2)\n    def average(values, mask):\n        return float(values[mask].mean()) if mask.any() else None\n    return dict(threshold=threshold, fixed_center_samples=int(center.sum()),\n        healthy_ink_pixels=int(healthy.sum()),\n        fixed_center_width_ratio_proxy=ratio,\n        healthy_ink_drop=average(np.maximum(original-output, 0), healthy),\n        healthy_ink_retention=average((output > threshold).astype(float), healthy),\n        healthy_ink_mse=average((original-output)**2, healthy),\n        reference_distance_on_fixed_union=average((output-ref)**2, union),\n        source_reference_distance_on_fixed_union=average((original-ref)**2, union))\n\n\ndef reference_sheet(conditions, size):\n    result = Image.new(\'RGB\', (size, size), \'white\')\n    half = size // 2\n    for i, c in enumerate(conditions):\n        glyph = Image.fromarray(np.rint((1-c[\'reference\'][0])*255).astype(np.uint8)).convert(\'RGB\')\n        result.paste(glyph.resize((half, half), Image.Resampling.BICUBIC), ((i % 2)*half, (i//2)*half))\n    return result\n\n\ndef width_probe_conditions(conditions):\n    """Optional +1 model-pixel dilation stress test, NOT a validated new font."""\n    return [dict(reference=maximum_filter(c[\'reference\'], size=(1, 3, 3)),\n                 strokes=maximum_filter(c[\'strokes\'], size=(1, 3, 3))) for c in conditions]\n\n\ndef run_case(pipe, extractor, case, *, steps=60, strength=.25, seed=2026,\n             cfg_scale=5., width_probe=False, progress=None,\n             boundary_guard=False, boundary_config=None, observed_width=False, width_config=None,\n             resume=False, on_arm_complete=None):\n    import torch\n    from diffusion_model.fitted_candidate_diffusion import image_tensor, optimize_clean\n    from stroke_model.restore_settle_erase import RestoreSettleErase\n    if steps < 4 or not 0 < strength < 1 or int(steps*strength) < 4:\n        raise ValueError(\'Need at least four actual repair steps\')\n    directory = Path(case[\'directory\'])\n    if observed_width and not boundary_guard:\n        raise ValueError(\'Observed-width comparison requires boundary_guard=True\')\n    out = directory / \'audit\'\n    out.mkdir(exist_ok=resume)\n    with Image.open(directory / \'input.png\') as im:\n        source = im.convert(\'RGB\').copy()\n    conditions = load_conditions(directory)\n    records = None\n    if resume:\n        from .resume import load_records, complete_arm, archive_partial, rgb_hash\n        if (out/\'input.png\').exists() and rgb_hash(out/\'input.png\') != rgb_hash(directory/\'input.png\'):\n            raise ValueError(\'Cached audit input differs from requested source\')\n        records = load_records(out, conditions)\n    if records is None:\n        records = fit_records(source, conditions, extractor, out / \'extraction_input\')\n    source_rgb = image_tensor(source, next(extractor.parameters()).device)\n    controller = RestoreSettleErase(source_rgb, extractor, records)\n    with torch.no_grad():\n        controller.refresh_erase(source_rgb)  # Diagnostic only; first restore step clears erasure.\n    support = fixed_support(source, controller)\n    np.savez_compressed(out / \'fixed_diagnostic_regions.npz\', **support)\n    # Reset the controller: diagnostic setup must not change the repair experiment.\n    controller = RestoreSettleErase(source_rgb, extractor, records)\n    reference = reference_sheet(conditions, source.width)\n    reference.save(out / \'reference_diagnostic_only.png\')\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor, return_dict=False)[0].float()/2+.5\n    def encode(rgb):\n        return pipe.vae.encode(rgb.float()*2-1).latent_dist.mode()*pipe.vae.config.scaling_factor\n    def save_tensor(rgb, path):\n        a = rgb[0].detach().permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(a*255).astype(np.uint8)).save(path)\n    # Same frozen float32 VAE and posterior-mode encoding for every compared arm.\n    # This is deliberately recorded: legacy cyclic used prepare_latents\' sampled posterior.\n    with torch.no_grad():\n        clean_input = encode(source_rgb)\n        save_tensor(decode(clean_input), out / \'vae_roundtrip.png\')\n        positive, negative = pipe.encode_prompt(case[\'prompt\'], \'cuda\', 1, True,\n                                               negative_prompt=case[\'negative_prompt\'])\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        noise = torch.randn(clean_input.shape, generator=torch.Generator(\'cpu\').manual_seed(seed),\n                            dtype=torch.float32).to(clean_input.device)\n        initial = pipe.scheduler.add_noise(clean_input, noise, timesteps[:1])\n    initial_hash = hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest()\n    legacy_initial_unrecorded = False\n    if resume:\n        signature = dict(steps=steps, strength=strength, seed=seed, cfg_scale=cfg_scale,\n            boundary_config=boundary_config, width_config=width_config,\n            input_rgb_sha256=rgb_hash(directory/\'input.png\'))\n        cache = out/\'resume_state.json\'\n        if cache.exists():\n            previous_state = json.loads(cache.read_text(encoding=\'utf-8\'))\n            if previous_state[\'signature\'] != signature:\n                raise ValueError(\'Resume settings/input differ from saved state\')\n            with np.load(out/\'resume_initial_latent.npz\', allow_pickle=False) as a:\n                initial = torch.as_tensor(a[\'initial\'].copy(), device=clean_input.device)\n            initial_hash = hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest()\n            if initial_hash != previous_state[\'initial_latent_sha256\']:\n                raise ValueError(\'Cached initial latent is damaged\')\n            legacy_initial_unrecorded = previous_state[\'legacy_initial_unrecorded\']\n        else:\n            old_report = out/\'report.json\'\n            if old_report.exists():\n                expected = json.loads(old_report.read_text(encoding=\'utf-8\'))[\'configuration\'][\'initial_latent_sha256\']\n                if expected != initial_hash:\n                    raise ValueError(\'Historical initial latent differs. Completed cases can be reused, but partial cases need matching runtime or a fresh comparison.\')\n            else:\n                legacy_initial_unrecorded = any(complete_arm(out, a, [int(t) for t in timesteps])\n                                                for a in (\'redenoise_only\',\'cyclic_guided\',\'boundary_guarded\',\'observed_width\'))\n            np.savez_compressed(out/\'resume_initial_latent.npz\', initial=initial.cpu().numpy())\n            json_write(cache, dict(signature=signature, initial_latent_sha256=initial_hash,\n                legacy_initial_unrecorded=legacy_initial_unrecorded))\n    # Expose reconstruction before the actual rerun so failures still retain evidence.\n    source.save(out / \'input.png\')\n    traces = {}\n    arms = [(\'redenoise_only\', None), (\'cyclic_guided\', controller)]\n    if boundary_guard:\n        from stroke_model.boundary_guarded_repair import BoundaryGuardConfig, BoundaryGuardedRepair\n        guard_config = BoundaryGuardConfig(**(boundary_config or {}))\n        guarded = BoundaryGuardedRepair(source_rgb, extractor, records, boundary_config=guard_config)\n        with torch.no_grad():\n            guarded.refresh_erase(source_rgb)\n        np.savez_compressed(out/\'boundary_initial_maps.npz\',\n            erase=guarded.erase.cpu().numpy(), restore=guarded.restore.cpu().numpy(),\n            preserve=guarded.preserve.cpu().numpy())\n        json_write(out/\'boundary_initial_diagnostics.json\', guarded.boundary_history)\n        guarded = BoundaryGuardedRepair(source_rgb, extractor, records, boundary_config=guard_config)\n        arms.append((\'boundary_guarded\', guarded))\n    if observed_width:\n        from stroke_model.observed_width_repair import ObservedWidthConfig, ObservedWidthRepair\n        def calibrated_controller():\n            return ObservedWidthRepair(source_rgb, extractor, records,\n                boundary_config=guard_config, width_config=ObservedWidthConfig(**(width_config or {})))\n        calibrated = calibrated_controller()\n        with torch.no_grad():\n            calibrated.refresh_erase(source_rgb)\n        np.savez_compressed(out/\'observed_width_initial_maps.npz\',\n            erase=calibrated.erase.cpu().numpy(), restore=calibrated.restore.cpu().numpy(),\n            preserve=calibrated.preserve.cpu().numpy())\n        json_write(out/\'observed_width_calibration.json\', calibrated.metadata)\n        json_write(out/\'observed_width_initial_diagnostics.json\', calibrated.boundary_history)\n        arms.append((\'observed_width\', calibrated_controller()))\n    if width_probe:\n        alternate = width_probe_conditions(conditions)\n        alt_records = fit_records(source, alternate, extractor, out / \'width_probe_input\')\n        arms.append((\'reference_width_probe\', RestoreSettleErase(source_rgb, extractor, alt_records)))\n    for arm, objective in arms:\n        if resume and complete_arm(out, arm, [int(t) for t in timesteps]):\n            traces[arm] = json.loads((out/(arm+\'_trace.json\')).read_text(encoding=\'utf-8\'))\n            if progress:\n                progress(case[\'case_id\'], arm, len(timesteps), len(timesteps),\n                         dict(phase=\'cached\', accepted_inner_steps=0))\n            continue\n        if resume:\n            archive_partial(out, arm)\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.clone()\n        trace = []\n        frames = out / arm\n        frames.mkdir()\n        for index, t in enumerate(timesteps):\n            with torch.no_grad():\n                model_input = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(model_input.to(dtype=next(pipe.unet.parameters()).dtype), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                base = scheduler.step((u+cfg_scale*(c-u)).float(), t, latent, eta=0., return_dict=True)\n            clean, latent = base.pred_original_sample.detach(), base.prev_sample.detach()\n            row = dict(step=index, timestep=int(t), phase=\'baseline\', accepted_inner_steps=0)\n            if objective is not None:\n                with torch.no_grad():\n                    row.update(objective.begin_step(decode(clean), index))\n                if row[\'phase\'] == \'restore\':\n                    clean, projection = objective.project_clean(clean, decode, encode)\n                    row.update(projection)\n                clean, optimization = optimize_clean(clean, decode, objective,\n                    iterations=2 if row[\'phase\'] in (\'restore\', \'settle\') else 4,\n                    step_ratio=.008, budget_ratio=.06, backtracks=5)\n                row.update(optimization)\n                with torch.no_grad():\n                    objective.finish_step(decode(clean), row)\n                previous = int(t) - scheduler.config.num_train_timesteps // scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[previous] if previous >= 0 else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent + alpha.sqrt() * (clean-base.pred_original_sample)\n                for name in (\'erase\', \'restore\', \'preserve\'):\n                    value = getattr(objective, name)[0, 0].detach().cpu().numpy()\n                    Image.fromarray(np.rint(value.clip(0, 1)*255).astype(np.uint8)).save(frames / f\'{name}_{index:03d}.png\')\n            with torch.no_grad():\n                save_tensor(decode(clean), frames / f\'clean_prediction_{index:03d}.png\')\n            latent = latent.detach()\n            trace.append(row)\n            if progress:\n                progress(case[\'case_id\'], arm, index+1, len(timesteps), row)\n        with torch.no_grad():\n            save_tensor(decode(latent), out / (arm+\'.png\'))\n        traces[arm] = trace\n        json_write(out / (arm+\'_trace.json\'), trace)\n        if objective is not None and hasattr(objective, \'boundary_history\'):\n            json_write(out/(arm+\'_map_history.json\'), objective.boundary_history)\n        if on_arm_complete:\n            on_arm_complete(case[\'case_id\'], arm)\n    results = {\'input\': source, \'vae_roundtrip\': Image.open(out / \'vae_roundtrip.png\').convert(\'RGB\')}\n    for arm, _ in arms:\n        results[arm] = Image.open(out / (arm+\'.png\')).convert(\'RGB\')\n    metric_rows = []\n    for arm, result in results.items():\n        # Save final extraction for VISUAL review; masks are not quality probabilities.\n        if arm != \'input\':\n            fit_records(result, conditions, extractor, out / (\'extraction_\'+arm), fit=False)\n        for slot in range(4):\n            half = source.width // 2\n            x, y = slot % 2 * half, slot // 2 * half\n            box = (x, y, x+half, y+half)\n            local = {k: v[y:y+half, x:x+half] for k, v in support.items()}\n            for threshold in (.35, .5, .65):\n                values = measure(source.crop(box), result.crop(box), reference.crop(box), local, threshold)\n                if boundary_guard:\n                    # Include all source foreground, including tips excluded by\n                    # the original automatic healthy mask. This still isn\'t GT.\n                    ink = ink_array(source.crop(box))\n                    all_source = dict(healthy=ink > .5, center=skeletonize(ink > .5))\n                    full = measure(source.crop(box), result.crop(box), reference.crop(box), all_source, threshold)\n                    values.update({\'all_source_\'+k:full[k] for k in\n                        (\'fixed_center_width_ratio_proxy\',\'healthy_ink_drop\',\'healthy_ink_retention\',\'healthy_ink_mse\')})\n                metric_rows.append(dict(case_id=case[\'case_id\'], kind=case[\'kind\'], slot=slot+1, arm=arm, **values))\n    json_write(out / \'report.json\', dict(case=case, metrics=metric_rows, traces=traces,\n        configuration=dict(steps=steps, actual_steps=len(timesteps), timesteps=[int(t) for t in timesteps],\n            strength=strength, seed=seed, cfg_scale=cfg_scale, vae_dtype=\'float32\',\n            vae_posterior=\'mode_shared_by_all_arms\' if not legacy_initial_unrecorded else \'legacy_initial_not_recorded\',\n            paired_initial_latent=not legacy_initial_unrecorded,\n            resume=resume, legacy_initial_unrecorded=legacy_initial_unrecorded,\n            initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n            width_probe=width_probe, original_repair_parameters=True,\n            boundary_guard=boundary_guard, boundary_config=boundary_config if boundary_guard else None,\n            observed_width=observed_width, width_config=width_config if observed_width else None,\n            legacy_difference=\'Posterior mode, not legacy stochastic VAE encoding\'),\n        limitations=[\'Widths are fixed-source-center distance proxies; movement is a confound.\',\n            \'Healthy masks are heuristic and require human review.\',\n            \'Reference similarity is not correctness and cannot prove template copying.\',\n            \'No automatic pass/fail or independent text accuracy claim.\',\n            \'Width probe is synthetic dilation and may be outside extractor training distribution.\']))\n    make_figures(out, source, reference, results, support)\n    return metric_rows\n\n\ndef make_figures(out, source, reference, results, support):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    labels = list(results)\n    fig, axes = plt.subplots(1, len(labels)+1, figsize=(4*(len(labels)+1), 4), squeeze=False)\n    for ax, (name, image) in zip(axes.flat, [(\'reference (diagnostic)\', reference), *results.items()]):\n        ax.imshow(image); ax.set_title(name); ax.axis(\'off\')\n    fig.tight_layout(); fig.savefig(out / \'comparison.png\', dpi=120); plt.close(fig)\n    fig, axes = plt.subplots(1, 3, figsize=(12, 4))\n    for ax, key in zip(axes, (\'healthy\', \'center\', \'initial_edit\')):\n        ax.imshow(source); ax.imshow(np.ma.masked_where(~support[key], support[key]), cmap=\'autumn\', alpha=.6)\n        ax.set_title(key+\' (heuristic)\'); ax.axis(\'off\')\n    fig.tight_layout(); fig.savefig(out / \'diagnostic_regions.png\', dpi=120); plt.close(fig)\n    for slot in range(1, 5):\n        with np.load(out / \'extraction_input\' / f\'slot_{slot}.npz\') as a:\n            count = len(a[\'extracted_masks\'])\n        fig, axes = plt.subplots(len(results), count+2,\n            figsize=(2.2*(count+2), 2.3*len(results)), squeeze=False)\n        half=source.width//2; x=(slot-1)%2*half; y=(slot-1)//2*half\n        for row, (arm, image) in enumerate(results.items()):\n            with np.load(out / (\'extraction_\'+arm) / f\'slot_{slot}.npz\') as a:\n                images = [(\'reference\', 1-a[\'reference\'][0]),\n                          (arm, np.asarray(image.crop((x,y,x+half,y+half))))] + [\n                    (f\'stroke {i+1}\', 1-m) for i,m in enumerate(a[\'extracted_masks\'])]\n            for ax, (name, value) in zip(axes[row], images):\n                ax.imshow(value, cmap=\'gray\', vmin=0, vmax=1) if value.ndim==2 else ax.imshow(value)\n                ax.set_title(name); ax.axis(\'off\')\n        fig.tight_layout(); fig.savefig(out / f\'extraction_slot_{slot}.png\', dpi=110); plt.close(fig)\n\n\ndef write_tables(root, cases, metrics, failures):\n    root = Path(root)\n    if metrics:\n        with (root / \'automatic_metrics.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(metrics[0]))\n            writer.writeheader(); writer.writerows(metrics)\n    fields = [\'case_id\', \'kind\', \'slot\', \'arm\', \'target_character\', \'input_error_type\',\n              \'extraction_ok\', \'error_reduced\', \'thinner\', \'new_error\', \'style_changed\', \'notes\']\n    with (root / \'human_review.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        for case in cases:\n            arms = sorted({r[\'arm\'] for r in metrics if r[\'case_id\'] == case[\'case_id\']})\n            for slot in range(1, 5):\n                for arm in arms:\n                    writer.writerow(dict(case_id=case[\'case_id\'], kind=case[\'kind\'], slot=slot, arm=arm))\n    json_write(root / \'failures.json\', failures)\n', 'repair_audit/resume.py': '"""Data-only archive recovery and completed-arm caching for stage 4.\n\nLegacy archives have no mid-arm latent/controller state. Restart partial arms;\nnever pretend that prediction PNGs are sampler checkpoints.\n"""\nfrom datetime import datetime\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .stage1 import read_archive, json_write\n\nARMS = (\'redenoise_only\', \'cyclic_guided\', \'boundary_guarded\', \'observed_width\')\n\n\ndef rgb_hash(path):\n    with Image.open(path) as image:\n        return hashlib.sha256(np.asarray(image.convert(\'RGB\')).tobytes()).hexdigest()\n\n\ndef prepare_resume(archive, output, expected_hashes):\n    """Copy only selected case data, never archived Python or absolute paths."""\n    with zipfile.ZipFile(archive) as z:\n        manifest = json.loads(z.read(\'runtime_manifest.json\'))\n        settings = manifest[\'settings\']\n        if not settings.get(\'boundary_config\') or not settings.get(\'width_config\'):\n            raise ValueError(\'Upload a stage-4 result ZIP, not a stage-3 input ZIP\')\n        if settings.get(\'reference_width_probe\'):\n            raise ValueError(\'Width-probe archives are not supported by this stage-4 resumer\')\n        # Audit/plot orchestration may evolve. All model and loss sources must match.\n        for name, expected in expected_hashes.items():\n            if name.startswith((\'stroke_model/\', \'diffusion_model/\')) and manifest[\'source_hashes\'].get(name) != expected:\n                raise ValueError(\'Repair implementation differs: \'+name)\n    cases, experiment = read_archive(archive, output, include_controls=True)\n    destination = Path(output)\n    with zipfile.ZipFile(archive) as z:\n        for case in cases:\n            prefix = case[\'case_id\']+\'/audit/\'\n            for entry in z.infolist():\n                name = entry.filename\n                if not name.startswith(prefix) or entry.is_dir():\n                    continue\n                relative = PurePosixPath(name)\n                if (\'\\\\\' in name or \':\' in name or relative.is_absolute()\n                        or \'..\' in relative.parts or relative.suffix not in (\'.png\',\'.npz\',\'.json\',\'.csv\')):\n                    continue\n                target = destination.joinpath(*relative.parts)\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(z.read(entry))\n            saved = Path(case[\'directory\'])/\'audit/input.png\'\n            if saved.exists() and rgb_hash(saved) != rgb_hash(Path(case[\'directory\'])/\'input.png\'):\n                raise ValueError(\'Cached audit input differs: \'+case[\'case_id\'])\n        for name in [\'human_review.csv\',\'automatic_metrics.csv\']:\n            if name in z.namelist():\n                (destination/name).write_bytes(z.read(name))\n    json_write(destination/\'resume_parent_manifest.json\', manifest)\n    return cases, experiment, manifest\n\n\ndef complete_arm(out, arm, timesteps):\n    out = Path(out)\n    png, trace = out/(arm+\'.png\'), out/(arm+\'_trace.json\')\n    if not png.is_file() or not trace.is_file():\n        return False\n    try:\n        with Image.open(png) as image:\n            dimensions = image.size\n            image.verify()\n        if (out/\'input.png\').exists():\n            with Image.open(out/\'input.png\') as source:\n                if source.size != dimensions:\n                    return False\n        rows = json.loads(trace.read_text(encoding=\'utf-8\'))\n        return (len(rows)==len(timesteps)\n                and [r[\'step\'] for r in rows]==list(range(len(timesteps)))\n                and [r[\'timestep\'] for r in rows]==list(timesteps))\n    except (ValueError, KeyError, OSError, TypeError):\n        return False\n\n\ndef archive_partial(out, arm):\n    """Keep incomplete artifacts before restarting; paths are fixed arm names."""\n    out = Path(out).resolve()\n    if arm not in ARMS:\n        raise ValueError(arm)\n    paths=[out/arm, out/(arm+\'.png\'), out/(arm+\'_trace.json\'), out/(arm+\'_map_history.json\')]\n    existing=[p for p in paths if p.exists()]\n    if not existing:\n        return\n    backup=out/\'interrupted_attempts\'/(arm+\'_\'+datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\'))\n    backup.mkdir(parents=True)\n    for path in existing:\n        if not path.resolve().is_relative_to(out) or not backup.resolve().is_relative_to(out):\n            raise ValueError(\'Unsafe backup path\')\n        path.rename(backup/path.name)\n\n\ndef load_records(out, conditions):\n    records=[]\n    try:\n        for slot,condition in enumerate(conditions,1):\n            path=Path(out)/\'extraction_input\'/f\'slot_{slot}.npz\'\n            with np.load(path,allow_pickle=False) as a:\n                if any(not np.array_equal(a[k],condition[k]) for k in (\'reference\',\'strokes\')):\n                    raise ValueError(\'Cached reference conditions differ\')\n                records.append(dict(reference=a[\'reference\'].copy(),strokes=a[\'strokes\'].copy(),\n                    fitted=a[\'fitted_strokes\'].copy(),extracted_masks=a[\'extracted_masks\'].copy(),\n                    fit_metadata=json.loads(path.with_name(f\'slot_{slot}_fit.json\').read_text(encoding=\'utf-8\'))))\n    except FileNotFoundError:\n        return None\n    return records\n\n\ndef completed_case(case, settings):\n    """A whole completed case can be reused without loading its sampler state."""\n    out=Path(case[\'directory\'])/\'audit\'\n    report=out/\'report.json\'\n    if not report.is_file():\n        return None\n    data=json.loads(report.read_text(encoding=\'utf-8\'))\n    cfg=data[\'configuration\']\n    for key in (\'steps\',\'strength\',\'seed\',\'boundary_config\',\'width_config\'):\n        if cfg.get(key)!=settings.get(key):\n            raise ValueError(\'Cached report settings differ: \'+key)\n    if not all(complete_arm(out,arm,cfg[\'timesteps\']) for arm in ARMS):\n        return None\n    return data[\'metrics\']\n', 'stroke_model/boundary_guarded_repair.py': '"""Experimental boundary uncertainty guard for the existing cyclic repair.\n\nOnly deletion-map construction changes. Extractor, fitted paths, restoration,\nloss weights, gradient step and DDIM transfer stay with RestoreSettleErase.\n"""\nfrom dataclasses import asdict, dataclass\nimport math\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom scipy.ndimage import binary_dilation\n\nfrom .restore_settle_erase import RestoreSettleErase\n\n\n@dataclass(frozen=True)\nclass BoundaryGuardConfig:\n    uncertainty_pixels: float = 1.\n    responsibility_floor: float = .5\n\n    def __post_init__(self):\n        if not math.isfinite(self.uncertainty_pixels) or self.uncertainty_pixels < 0:\n            raise ValueError(\'uncertainty_pixels must be finite and nonnegative\')\n        if not math.isfinite(self.responsibility_floor) or not 0 <= self.responsibility_floor < 1:\n            raise ValueError(\'responsibility_floor must be in [0,1)\')\n\n\nclass BoundaryGuardedRepair(RestoreSettleErase):\n    def __init__(self, *args, boundary_config=None, **kwargs):\n        self.boundary_config = boundary_config or BoundaryGuardConfig()\n        self.boundary_history = []\n        super().__init__(*args, **kwargs)\n        self.metadata.update(kind=\'restore_settle_erase_boundary_guard_v1\',\n                             boundary_guard=asdict(self.boundary_config))\n\n    @torch.no_grad()\n    def refresh_erase(self, image):\n        erase = torch.zeros_like(self.erase)\n        rows = []\n        cfg = self.boundary_config\n        for slot, r in enumerate(self.records, 1):\n            ink = self.crop(self.ink(image).clamp(0, 1), r[\'box\'], r[\'size\'])\n            p = self.extractor(ink, r[\'reference\'], r[\'strokes\'])[\'probabilities\'][0].float().cpu().numpy()\n            if not np.isfinite(p).all() or p.shape != r[\'distances\'].shape:\n                raise ValueError(\'Invalid stroke extraction\')\n            # The paths and their calibrated radii stay fixed. This is an\n            # uncertainty allowance, not a new target thickness or wider stroke.\n            radius = r[\'radii\'] + cfg.uncertainty_pixels\n            outside = np.clip((r[\'distances\'] - radius - 1.5) / 1.5, 0, 1)\n            safe = (r[\'distances\'] <= radius + 1.).any(0)\n            responsibility = (p * outside).sum(0)\n            observed = ink[0, 0].cpu().numpy()\n            eligible = (responsibility > cfg.responsibility_floor) & ~safe\n            seeds = (observed > .15) & eligible\n            grown = binary_dilation(seeds, iterations=1)\n            # Recheck every grown pixel. A faint boundary seed must not turn an\n            # otherwise ineligible neighboring body pixel into a deletion target.\n            weight = grown * responsibility * eligible\n            patch = self.tensor(weight)\n            gate = self.tensor(eligible.astype(np.float32))\n            x0, y0, x1, y1 = r[\'box\']\n            shape = (y1-y0, x1-x0)\n            projected = F.interpolate(patch, shape, mode=\'bilinear\', align_corners=False)\n            gate = F.interpolate(gate, shape, mode=\'nearest\') > .5\n            erase[..., y0:y1, x0:x1] = projected * gate\n            rows.append(dict(slot=slot, seeds=int(seeds.sum()),\n                black_seeds=int((seeds & (observed > .5)).sum()),\n                grown_rejected=int((grown & ~eligible).sum()),\n                deletion_pixels_model=int((weight > .01).sum())))\n        self.erase = erase * (self.restore <= 0).float() * (1-self.path_union)\n        self.update_support()\n        self.boundary_history.append(dict(refresh=len(self.boundary_history), slots=rows,\n            sheet_deletion_pixels=int((self.erase > .01).sum())))\n', 'stroke_model/observed_width_repair.py': '"""Calibrate deletion corridors of closed fallback strokes from original ink.\n\nOnly deletion radii change. Paths, restoration, fitting and the image itself do\nnot change. Robust samples near the expected path limit the influence of spurs.\n"""\nfrom dataclasses import asdict, dataclass\nimport math\nimport numpy as np\nfrom scipy.ndimage import distance_transform_edt\nfrom skimage.morphology import skeletonize\n\nfrom .boundary_guarded_repair import BoundaryGuardedRepair\n\n\n@dataclass(frozen=True)\nclass ObservedWidthConfig:\n    minimum_samples: int = 12\n    minimum_path_coverage: float = .7\n    maximum_increase: float = 4.\n    maximum_half_width: float = 8.\n\n    def __post_init__(self):\n        if type(self.minimum_samples) is not int or self.minimum_samples < 3:\n            raise ValueError(\'minimum_samples must be an integer >=3\')\n        if not math.isfinite(self.minimum_path_coverage) or not 0 < self.minimum_path_coverage <= 1:\n            raise ValueError(\'minimum_path_coverage must be in (0,1]\')\n        for value in (self.maximum_increase, self.maximum_half_width):\n            if not math.isfinite(value) or value <= 0:\n                raise ValueError(\'Width limits must be finite and positive\')\n\n\nclass ObservedWidthRepair(BoundaryGuardedRepair):\n    def __init__(self, source, extractor, records, *, width_config=None, **kwargs):\n        self.width_config = width_config or ObservedWidthConfig()\n        super().__init__(source, extractor, records, **kwargs)\n        cfg = self.width_config\n        self.width_calibration = []\n        # Calibrate once from the untouched source, never from later predictions.\n        for slot, (record, raw) in enumerate(zip(self.records, records), 1):\n            for k, info in enumerate(raw.get(\'fit_metadata\', [{} for _ in raw[\'strokes\']])):\n                old = float(record[\'radii\'][k, 0, 0])\n                row = dict(slot=slot, stroke=k+1, old_radius=old, new_radius=old,\n                           method=info.get(\'method\'), reason=\'not_closed_fallback\')\n                if (info.get(\'reference_path\', {}).get(\'closed\')\n                        and \'fallback\' in info.get(\'method\', \'\')):\n                    path = record[\'distances\'][k] == 0\n                    observed = raw[\'extracted_masks\'][k] > .35\n                    coverage = int((path & observed).sum()) / max(int(path.sum()), 1)\n                    # Measure at observed centers, anchored near the expected loop.\n                    # Sampling on a slightly offset prototype underestimates width.\n                    centers = skeletonize(observed)\n                    supported = centers & (record[\'distances\'][k] <= old+cfg.maximum_increase)\n                    count = int(supported.sum())\n                    row.update(samples=count, path_coverage=coverage)\n                    if count >= cfg.minimum_samples and coverage >= cfg.minimum_path_coverage:\n                        values = distance_transform_edt(observed)[supported]\n                        estimate = float(np.median(values))\n                        new = max(old, min(estimate, old+cfg.maximum_increase, cfg.maximum_half_width))\n                        record[\'radii\'][k, 0, 0] = new\n                        row.update(observed_half_width=estimate, new_radius=new,\n                                   reason=\'calibrated\' if new > old else \'no_increase_needed\')\n                    else:\n                        row[\'reason\'] = \'insufficient_visible_path_support\'\n                self.width_calibration.append(row)\n        self.metadata.update(kind=\'closed_fallback_observed_width_v1\',\n                             observed_width_config=asdict(cfg), width_calibration=self.width_calibration)\n', 'repair_audit/stage3.py': '"""Expand the frozen stage-2 comparison to new reference glyphs.\n\nThis evaluates new glyphs from the existing training catalog, not unseen-font\ngeneralization. Shape judgments remain manual; no template straightening is used.\n"""\nimport csv\nimport json\nfrom pathlib import Path\nimport shutil\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nfrom .stage1 import file_hash, json_write, write_tables\n\n\ndef choose_rows(catalog, count=12, excluded_rows=(1, 2, 3, 5)):\n    """Deterministically spread choices across the catalog, without quality filtering."""\n    excluded_hashes = {r[\'reference_sha256\'] for r in catalog\n                       if r[\'manifest_row\'] in excluded_rows}\n    candidates = [r[\'manifest_row\'] for r in catalog\n                  if r[\'manifest_row\'] not in excluded_rows\n                  and r[\'reference_sha256\'] not in excluded_hashes]\n    if type(count) is not int or count < 4 or count % 4 or len(candidates) < count:\n        raise ValueError(\'Need a multiple of four new glyphs (default 12), available: \'+str(len(candidates)))\n    return [candidates[i] for i in np.linspace(0, len(candidates)-1, count, dtype=int)]\n\n\ndef validate_rows(catalog, rows, excluded_rows=(1, 2, 3, 5)):\n    by_row = {r[\'manifest_row\']: r for r in catalog}\n    excluded_hashes = {r[\'reference_sha256\'] for r in catalog\n                       if r[\'manifest_row\'] in excluded_rows}\n    if (not rows or len(rows) % 4 or len(set(rows)) != len(rows)\n            or any(type(r) is not int or r not in by_row for r in rows)):\n        raise ValueError(\'Choose distinct catalog row IDs, in groups of four.\')\n    hashes = [by_row[r][\'reference_sha256\'] for r in rows]\n    if any(r in excluded_rows for r in rows) or excluded_hashes.intersection(hashes):\n        raise ValueError(\'Exclude the four previously inspected glyphs (rows 1,2,3,5 and matching contents).\')\n    if len(set(hashes)) != len(hashes):\n        raise ValueError(\'Duplicate reference contents; select different glyphs.\')\n    return [rows[i:i+4] for i in range(0, len(rows), 4)]\n\n\ndef materialize_inputs(groups, output, checkpoint_sha256):\n    """Keep every generated page and every normal control; record failed generation."""\n    from diffusion_model.multiglyph_pilot import compose_sheet\n    from diffusion_model.score_pilot import training_catalog\n    output = Path(output)\n    cases, failures = [], []\n    for group in groups:\n        root = Path(group[\'root\'])\n        entries = training_catalog(group[\'manifest\'])\n        normal, _ = compose_sheet(entries, 576)\n        report_path = root/\'generation/generation_report.json\'\n        report = json.loads(report_path.read_text(encoding=\'utf-8\')) if report_path.is_file() else {}\n        failure_path = root/\'generation_failure.json\'\n        if failure_path.is_file():\n            failures.append(dict(case_id=group[\'group_id\'], stage=\'generation\',\n                                 detail=json.loads(failure_path.read_text(encoding=\'utf-8\'))))\n        pages = [p for p in report.get(\'pages\', []) if not p.get(\'control\')]\n        if not pages and not failure_path.is_file():\n            failures.append(dict(case_id=group[\'group_id\'], stage=\'generation\', error=\'No generated pages\'))\n        sources = [(group[\'group_id\']+\'_normal_control\', normal, \'normal_control\', {})]\n        for page in pages:\n            identifier = f"{group[\'group_id\']}_page_{page[\'page_id\']:03d}"\n            if page.get(\'safety_filtered\'):\n                failures.append(dict(case_id=identifier, stage=\'generation\', error=\'safety_filtered\', page=page))\n                continue\n            # The generator writes this named member itself; ignore report absolute paths.\n            with Image.open(root/\'generation\'/f"page_{page[\'page_id\']:03d}.png") as image:\n                sources.append((identifier, image.convert(\'RGB\').copy(), \'generated\', page))\n        for identifier, image, kind, page in sources:\n            target = output/identifier\n            target.mkdir(exist_ok=False)\n            image.save(target/\'input.png\')\n            for slot in range(1, 5):\n                shutil.copy2(root/\'source_conditions\'/f\'condition_{slot}.npz\', target/f\'condition_{slot}.npz\')\n            case = dict(case_id=identifier, directory=str(target), kind=kind,\n                        prompt=report.get(\'prompts\', {}).get(\'4\',\n                            \'a typography sheet with 4 Korean Hangul glyphs arranged in \'\n                            \'2 rows and 2 columns, black ink on plain white background, \'\n                            \'flat front view, no grid lines, preserve the glyphs and layout of the input image\'),\n                        negative_prompt=report.get(\'negative_prompt\',\n                            \'photograph, scenery, paper texture, shadows, colored background, grid lines, decorations\'),\n                        generation_seed=page.get(\'seed\'), generation_strength=page.get(\'strength\'),\n                        checkpoint_sha256=checkpoint_sha256, source_sha256=file_hash(target/\'input.png\'),\n                        original_rows=group[\'original_rows\'], char_ids=group[\'char_ids\'])\n            json_write(target/\'input_metadata.json\', case)\n            cases.append(case)\n    return cases, failures\n\n\ndef shape_comparison(case, arms=None):\n    """Fixed slot crops: no OCR selection, contour smoothing or straightening."""\n    out = Path(case[\'directory\'])/\'audit\'\n    arms = arms or [\'input\', \'vae_roundtrip\', \'redenoise_only\', \'cyclic_guided\', \'boundary_guarded\']\n    if not all((out/f\'{arm}.png\').is_file() for arm in arms):\n        return\n    images = []\n    for arm in arms:\n        with Image.open(out/f\'{arm}.png\') as image:\n            images.append(image.convert(\'RGB\'))\n    cell, margin, heading = 192, 24, 38\n    canvas = Image.new(\'RGB\', (cell*len(arms), heading+(cell+margin)*4), \'white\')\n    draw = ImageDraw.Draw(canvas)\n    for col, arm in enumerate(arms):\n        draw.text((col*cell+5, 12), arm, fill=\'black\')\n    for index in range(4):\n        row = case.get(\'original_rows\', [\'?\']*4)[index]\n        y = heading+index*(cell+margin)\n        draw.text((5, y+3), f\'slot {index+1} / original row {row}\', fill=\'black\')\n        for col, image in enumerate(images):\n            half = image.width//2\n            x0, y0 = (index % 2)*half, (index//2)*half\n            crop = image.crop((x0, y0, x0+half, y0+half))\n            canvas.paste(crop.resize((cell, cell), Image.Resampling.LANCZOS), (col*cell, y+margin))\n    canvas.save(out/\'shape_comparison.png\')\n\n\ndef write_stage3_tables(root, cases, metrics, failures):\n    """Extend review questions and preserve any manually filled answers on rerun."""\n    root = Path(root)\n    review = root/\'human_review.csv\'\n    previous = {}\n    if review.is_file():\n        with review.open(encoding=\'utf-8-sig\', newline=\'\') as stream:\n            previous = {(r[\'case_id\'], r[\'slot\'], r[\'arm\']): r for r in csv.DictReader(stream)}\n    write_tables(root, cases, metrics, failures)\n    with review.open(encoding=\'utf-8-sig\', newline=\'\') as stream:\n        reader = csv.DictReader(stream)\n        fields = list(reader.fieldnames)\n        rows = list(reader)\n    fields += [\'original_manifest_row\', \'catalog_char_id\', \'input_has_error\',\n               \'straight_or_curve_worsened\', \'endpoint_deformed\', \'junction_deformed\', \'residual_artifact\']\n    by_id = {c[\'case_id\']: c for c in cases}\n    for row in rows:\n        key = (row[\'case_id\'], row[\'slot\'], row[\'arm\'])\n        row.update({k: v for k, v in previous.get(key, {}).items() if k in fields and v})\n        case, slot = by_id[row[\'case_id\']], int(row[\'slot\'])-1\n        row[\'original_manifest_row\'] = case[\'original_rows\'][slot]\n        row[\'catalog_char_id\'] = case[\'char_ids\'][slot]\n        if case[\'kind\'] == \'normal_control\':\n            row[\'input_has_error\'] = \'no\'\n            row[\'error_reduced\'] = \'n/a\'\n    with review.open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader(); writer.writerows(rows)\n'}
SOURCE_HASHES={'stroke_model/reference_model.py': 'df3445a74e2543ee9e86037b03316650918231f99da41dcf1efb0fb81b7f597f', 'stroke_model/stroke_reference_fit.py': '2c8cc24abcf3dfb555eff5622b63e5ab83460f802e39e36a706f38f2c96c89e1', 'diffusion_model/fitted_candidate_diffusion.py': '5fd3422b7b94a7443cbb639253b73e028744075ad3f55f2e7fab7511a77e8f0f', 'stroke_model/restore_settle_erase.py': '2823d83a9c1b6d9fb0a5a1861d5e0ea56af39edc60c526231a7b571c6d5fc575', 'repair_audit/stage1.py': '83a898997f8b75a68ffff25675a5903064465fe7ad3188e4a9a70d549488a94f', 'repair_audit/resume.py': 'ec00101b351c1738fc9c546eb2fbc100c39a020f8c0c5c5be4c0cd757f33a638', 'stroke_model/boundary_guarded_repair.py': '5c1f69340934c1be4096d5cc331bb05bdee73e8728a594b92913c5159bd09484', 'stroke_model/observed_width_repair.py': '60efd3e8c151835983c2e65b598098b90f89d076b4c1c72dd46f4c3de3e30f37', 'repair_audit/stage3.py': '3a7cbb12d83ac9930cfed68b810e9acbf67f15fcca1dfa154d5d1431ba148df6'}

ROOT = Path('/content/hsqm_stage4_resume_runtime')
ROOT.mkdir(exist_ok=True)
for package in ('stroke_model', 'diffusion_model', 'repair_audit'):
    (ROOT/package).mkdir(exist_ok=True)
    (ROOT/package/'__init__.py').touch()
for name, source in RUNTIME_HELPERS.items():
    path = ROOT/name
    path.write_text(source, encoding='utf-8')
    if hashlib.sha256(path.read_bytes()).hexdigest() != SOURCE_HASHES[name]:
        raise RuntimeError('Embedded source hash mismatch: '+name)
sys.path.insert(0, str(ROOT))
for name in list(sys.modules):
    if name.split('.')[0] in ('stroke_model', 'diffusion_model', 'repair_audit'):
        sys.modules.pop(name)
import torch
from PIL import Image
from IPython.display import display
from google.colab import drive, files
from repair_audit.stage1 import *
if not torch.cuda.is_available():
    raise RuntimeError('런타임 유형을 GPU(L4 권장)로 바꾼 뒤 새 세션에서 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('진단 코드 준비 완료. 다음 셀에서 기존 결과 ZIP과 best.pt를 지정하세요.')
from repair_audit.stage3 import shape_comparison, write_stage3_tables
from repair_audit.resume import prepare_resume, completed_case, complete_arm, ARMS

## 2번: 중단한 결과 ZIP 선택
- 3단계 입력 ZIP 대신 **중단한 4단계 결과 ZIP** 또는 이 재개 코랩의 checkpoint ZIP을 선택합니다.
- 보정 설정·모델 revision은 ZIP에서 복구하며 모델/손실 소스와 체크포인트 해시를 검사합니다.
- 완료 여부는 최종 PNG와 전체 timestep trace를 함께 확인합니다. 중간 PNG만으로 완료 처리하지 않습니다.
- partial 경로의 예전 프레임은 interrupted_attempts/로 보관합니다.
- Drive 자동 백업 위치: MyDrive/HSQM_stage4_resume_results.

In [ ]:
# 2. 중단한 4단계 ZIP 업로드. 설정은 ZIP에서 복구합니다.
ARCHIVE = ''
CHECKPOINT = '/content/drive/Shareddrives/2026 자율연구/HSQM/v2_topology_runs/20261104_214705_208224/best.pt'
DRIVE_RESULT_DIR = '/content/drive/MyDrive/HSQM_stage4_resume_results'
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
if not ARCHIVE:
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise ValueError('중단한 결과 ZIP 하나만 선택하세요.')
    ARCHIVE=str(Path(next(iter(uploaded))).resolve())
    del uploaded
OUTPUT=Path('/content')/('repair_stage4_resume_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
CASES, EXPERIMENT, PARENT = prepare_resume(ARCHIVE, OUTPUT, SOURCE_HASHES)
if file_hash(CHECKPOINT)!=PARENT['checkpoint_sha256']:
    raise ValueError('기존 실행과 다른 best.pt입니다.')
MODEL_ID, REVISION = PARENT['model_id'], PARENT['revision']
SETTINGS = PARENT['settings']
REPAIR_STEPS, REPAIR_STRENGTH, REPAIR_SEED = [SETTINGS[k] for k in ('steps','strength','seed')]
BOUNDARY_CONFIG, WIDTH_CONFIG = SETTINGS['boundary_config'], SETTINGS['width_config']
json_write(OUTPUT/'runtime_manifest.json', dict(source_hashes=SOURCE_HASHES,
    model_id=MODEL_ID, revision=REVISION, checkpoint_sha256=PARENT['checkpoint_sha256'],
    torch_version=torch.__version__, gpu_type=torch.cuda.get_device_name(0),
    settings=SETTINGS, resume_archive=str(ARCHIVE)))
METRICS, FAILURES = [], []
for case in CASES:
    cached=completed_case(case, SETTINGS)
    if cached is not None:
        METRICS.extend(cached)
    audit=Path(case['directory'])/'audit'
    done=[a for a in ARMS if complete_arm(audit,a,list(range(225,-1,-16)))] if REPAIR_STEPS==60 and REPAIR_STRENGTH==.25 else []
    print(case['case_id'], '사례 전체 재사용' if cached is not None else '완료 경로: '+str(done))

# Own timestamped ZIP only; each completed arm refreshes this recovery snapshot.
def save_snapshot():
    write_stage3_tables(OUTPUT, CASES, METRICS, FAILURES)
    archive=shutil.make_archive(str(OUTPUT)+'_checkpoint','zip',OUTPUT)
    if DRIVE_RESULT_DIR:
        destination=Path(DRIVE_RESULT_DIR)
        destination.mkdir(parents=True,exist_ok=True)
        shutil.copy2(archive, destination/Path(archive).name)
    return archive
def arm_saved(case_id, arm):
    archive=save_snapshot()
    print('경로 완료·Drive 백업:',case_id,arm,Path(archive).name,flush=True)
save_snapshot()
print('기존 결과 복구 및 Drive 백업 완료. 3번 모델 로딩 → 4번 실행.')

In [ ]:
# 3. 기존 획 추출기와 SD 로딩 (학습 안 함)
from stroke_model.reference_model import ReferenceStrokeModel
from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
if state.get('architecture') != 'reference_v2_topology_loss_v1':
    raise ValueError('이 진단은 기존 reference_v2_topology_loss_v1 체크포인트용입니다.')
extractor = ReferenceStrokeModel(image_size=state['image_size'], width=state['width'], pretrained=False)
extractor.load_state_dict(state['model'], strict=True)
for case in CASES:
    if any(c['reference'].shape[-2:] != (state['image_size'], state['image_size'])
           for c in load_conditions(case['directory'])):
        raise ValueError('조건 해상도와 체크포인트 해상도가 다릅니다.')
extractor = extractor.eval().requires_grad_(False).to('cuda')
del state; gc.collect(); torch.cuda.empty_cache()
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(MODEL_ID, revision=REVISION,
    torch_dtype=torch.float16, use_safetensors=True)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, clip_sample=False, thresholding=False)
pipe.to('cuda'); pipe.enable_attention_slicing(); pipe.enable_vae_slicing()
for module in (pipe.unet, pipe.vae, pipe.text_encoder):
    module.eval().requires_grad_(False)
pipe.vae.to(dtype=torch.float32)
json_write(OUTPUT/'scheduler_config.json', dict(pipe.scheduler.config))
import importlib.metadata
json_write(OUTPUT/'versions.json', {name: importlib.metadata.version(name) for name in
    ('torch','diffusers','transformers','timm','numpy','scipy','scikit-image')})
print('로딩 완료. 4번 셀을 실행하세요.')

In [ ]:
# 4. 완료 경로 재사용, 부분 경로는 처음부터 재계산
def progress(case_id, arm, step, total, row):
    if row['phase']=='cached' or step==1 or step%3==0 or step==total:
        print(case_id,arm,f'{step}/{total}',row['phase'],
              'accepted=',row['accepted_inner_steps'],flush=True)
for case in CASES:
    try:
        cached=completed_case(case, SETTINGS)
        if cached is None:
            rows=run_case(pipe,extractor,case,steps=REPAIR_STEPS,strength=REPAIR_STRENGTH,
                seed=REPAIR_SEED,boundary_guard=True,boundary_config=BOUNDARY_CONFIG,
                observed_width=True,width_config=WIDTH_CONFIG,resume=True,
                progress=progress,on_arm_complete=arm_saved)
        else:
            rows=cached
            print('SKIP 완료 사례:',case['case_id'],flush=True)
        METRICS[:]=[r for r in METRICS if r['case_id']!=case['case_id']]
        METRICS.extend(rows)
        FAILURES[:]=[f for f in FAILURES if f.get('case_id')!=case['case_id']]
        shape_comparison(case,arms=['input','vae_roundtrip','redenoise_only','cyclic_guided','boundary_guarded','observed_width'])
        display(Image.open(Path(case['directory'])/'audit/shape_comparison.png'))
    except KeyboardInterrupt:
        print('중단 요청: 현재 부분 경로는 다음에 처음부터 계산합니다. 백업 후 종료합니다.',flush=True)
        break
    except Exception as exc:
        failure=dict(case_id=case['case_id'],type=type(exc).__name__,error=str(exc),traceback=traceback.format_exc())
        FAILURES.append(failure)
        json_write(Path(case['directory'])/'resume_failure.json',failure)
        print('FAILED:',case['case_id'],str(exc),flush=True)
    finally:
        save_snapshot()
        gc.collect(); torch.cuda.empty_cache()
print('완료 또는 중단 상태를 Drive에 저장했습니다. 5번에서 ZIP 다운로드.')

## 결과 검토
- shape_comparison.png: 글자별 6개 결과 비교. boundary_guarded와 observed_width가 핵심 비교입니다.
- 정상 row148의 ㅇ, row202의 ㅇ와 row64: 굵기·윤곽 보존이 좋아지는지 확인합니다.
- row131 등 적용 대상이 아닌 획의 변형도 남는지 확인합니다.
- 생성 입력: 돌출·잡티 제거가 유지되는지 반드시 함께 봅니다.
- observed_width_calibration.json: 획별 기존/새 반경, 샘플 수, 적용/거절 사유.
- observed_width_initial_maps.npz와 boundary_initial_maps.npz: 원본 초기 삭제 지도 비교.
- 각 arm/erase_*.png, restore_*.png와 *_trace.json: 삭제 침범·복원 투영 기록.
- human_review.csv: 오류 감소·얇아짐·직선/곡선·끝점·접합부·잔상을 각각 판정합니다.
- 자동 폭 비율은 실제 굵기 변화율이나 정답률이 아닙니다.

5번 셀 결과 ZIP을 보내주세요. 초기 지도 개선과 최종 GPU 품질은 구분해서 판단합니다.

In [ ]:
# 4. 완료 경로 재사용, 부분 경로는 처음부터 재계산
def progress(case_id, arm, step, total, row):
    if row['phase']=='cached' or step==1 or step%3==0 or step==total:
        print(case_id,arm,f'{step}/{total}',row['phase'],
              'accepted=',row['accepted_inner_steps'],flush=True)
for case in CASES:
    try:
        cached=completed_case(case, SETTINGS)
        if cached is None:
            rows=run_case(pipe,extractor,case,steps=REPAIR_STEPS,strength=REPAIR_STRENGTH,
                seed=REPAIR_SEED,boundary_guard=True,boundary_config=BOUNDARY_CONFIG,
                observed_width=True,width_config=WIDTH_CONFIG,resume=True,
                progress=progress,on_arm_complete=arm_saved)
        else:
            rows=cached
            print('SKIP 완료 사례:',case['case_id'],flush=True)
        METRICS[:]=[r for r in METRICS if r['case_id']!=case['case_id']]
        METRICS.extend(rows)
        FAILURES[:]=[f for f in FAILURES if f.get('case_id')!=case['case_id']]
        shape_comparison(case,arms=['input','vae_roundtrip','redenoise_only','cyclic_guided','boundary_guarded','observed_width'])
        display(Image.open(Path(case['directory'])/'audit/shape_comparison.png'))
    except KeyboardInterrupt:
        print('중단 요청: 현재 부분 경로는 다음에 처음부터 계산합니다. 백업 후 종료합니다.',flush=True)
        break
    except Exception as exc:
        failure=dict(case_id=case['case_id'],type=type(exc).__name__,error=str(exc),traceback=traceback.format_exc())
        FAILURES.append(failure)
        json_write(Path(case['directory'])/'resume_failure.json',failure)
        print('FAILED:',case['case_id'],str(exc),flush=True)
    finally:
        save_snapshot()
        gc.collect(); torch.cuda.empty_cache()
print('완료 또는 중단 상태를 Drive에 저장했습니다. 5번에서 ZIP 다운로드.')